In [ ]:
import os
import re
import json
import glob
import asyncio
import requests
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional, Iterable
from datetime import datetime
from collections import defaultdict
from tabulate import tabulate
from rank_bm25 import BM25Okapi
from time import perf_counter
import hashlib
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

import chromadb  # pure chroma
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama  # hanya LLM-nya aja

# MCP client
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# --- TOTAL TIMING HELPERS (ADD) ---
def _now() -> float:
    return perf_counter()

def _elapsed_ms(t0: float) -> int:
    return int((perf_counter() - t0) * 1000)

def _log_total(label: str, t0: float):
    ms = _elapsed_ms(t0)
    print(f"\n[TOTAL] {label}: {ms} ms (~{ms/1000:.3f}s)")


def _print_banner(title: str, width: int = 150) -> str:
    """
    Cetak banner dengan garis horizontal dan judul di tengah.
    Return: line string yang bisa dipakai lagi kalau mau.
    """
    line = "─" * width
    print(f"\n{line}")
    print(title.center(width))
    print(line)
    return line

# ----------------------------------

# =========================
# === CONFIG (ENV) ========
# =========================
JSON_INPUT       = os.getenv("EXPORT_JSONL_OUTPUT", "./datahub_metadata/*.jsonl").strip()
CHROMA_DIR       = os.getenv("CHROMA_DIR", "./chroma_db")
COLLECTION_NAME  = os.getenv("COLLECTION_NAME", "classicmodels")
OLLAMA_BASE_URL  = os.getenv("OLLAMA_URL", "http://localhost:11434")
CHAT_MODEL       = os.getenv("CHAT_MODEL", "HridaAI/hrida-t2sql:q8_0")
TOP_K            = int(os.getenv("TOP_K", "5"))

MICRO_PROBE = ["id", "date", "status"]

SPARSE_BONUS     = float(os.getenv("SPARSE_BONUS", "0.75"))
NAME_QUERY_BONUS = float(os.getenv("NAME_QUERY_BONUS", "0.3"))
ID_QUERY_BONUS   = float(os.getenv("ID_QUERY_BONUS", "0.4"))

# >>> SPEED: batasi berapa banyak dokumen yang masuk ke context LLM
MAX_CTX_DOCS = int(os.getenv("MAX_CTX_DOCS", "12"))
SPARSE_MAX_HITS  = int(os.getenv("SPARSE_MAX_HITS", "20"))

# bisa dimatiin kalau mau lebih silent
SPARSE_DEBUG   = bool(int(os.getenv("SPARSE_DEBUG", "0")))
RETRIEVE_DEBUG = bool(int(os.getenv("RETRIEVE_DEBUG", "1")))

COLUMN_PRIORITY_CONFIG: List[Dict[str, Any]] = [
    {
        "id": "customers.primary_name",
        "tables": ["customers"],
        "role_any": ["name"],
        "alias_any": [
            "customer name",
            "nama customer",
            "nama pelanggan",
            "company name",
            "nama perusahaan",
            "account name",
        ],
        "column_contains": ["customername"],
        "query_terms_any": ["name", "nama"],
        "query_terms_also_any": ["customer", "customers", "pelanggan"],
        "boost": 0.4,
    },
]

MCP_BASE    = os.getenv("MCP_BASE", "http://mcp_server:8000/mcp")
MCP_TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", "60"))

SQL_DEBUG = int(os.getenv("SQL_DEBUG", "1"))
def _dbg_print_two(label: str, sql: str):
    if SQL_DEBUG:
        print(f"\n{label}")
        print(sql)

# =========================
# === RAG (STATE) =========
# =========================
# =========================
# === RAG (STATE) =========
# =========================
class RAGState(TypedDict):
    jsonl_paths: List[str]
    question: str
    added: int
    updated: int
    skipped: int
    context: str
    answer: str
    draft_sql: str
    final_sql: str

    # NEW: jenis query hasil classifier ('lookup', 'aggregate_by', etc.)
    qtype: str

    # NEW: flag apakah pertanyaan ini butuh REGEXP/pattern handling
    requires_regex: bool

    # NEW: catatan dari LLM (Agent 1/2) tentang alasan join, dll.
    notes: str

    # NEW: daftar error whitelist (identifier tidak ada di ALLOWED IDENTIFIERS)
    validation_errors: List[str]

    # NEW: ringkasan apa saja yang dikoreksi Agent 2 (join/regex/whitelist)
    critic_feedback: str

# =========================
# === UTIL ================
# =========================
def resolve_jsonl_paths(spec: str) -> List[str]:
    paths: List[str] = []
    if not spec:
        return paths
    p = Path(spec)
    if any(ch in spec for ch in "*?[]"):
        paths.extend(glob.glob(spec))
    elif p.is_dir():
        paths.extend(str(x) for x in p.glob("*.jsonl"))
    elif p.is_file():
        paths.append(str(p))
    else:
        paths.extend(glob.glob(spec))
    out, seen = [], set()
    for x in paths:
        xp = str(Path(x))
        if xp.lower().endswith(".jsonl") and xp not in seen:
            seen.add(xp)
            out.append(xp)
    return sorted(out)

# --- FIXED: OllamaEmbeddingFunction (indent + clean config methods) ---
class OllamaEmbeddingFunction(EmbeddingFunction):
    def __init__(self, base_url: str, model: str = "bge-m3:latest", timeout: int = 60):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.timeout = int(timeout)

    def __call__(self, texts: Documents) -> Embeddings:
        out: Embeddings = []
        for t in texts:
            resp = requests.post(
                f"{self.base_url}/api/embeddings",
                json={"model": self.model, "prompt": t},
                timeout=self.timeout,
            )
            resp.raise_for_status()
            out.append(resp.json()["embedding"])
        return out

    @classmethod
    def name(cls) -> str:
        return "ollama"

    @classmethod
    def build_from_config(cls, config: dict) -> "OllamaEmbeddingFunction":
        return cls(
            base_url=str(config.get("base_url", "http://localhost:11434")),
            model=str(config.get("model", "bge-m3:latest")),
            timeout=int(config.get("timeout", 60)),
        )

    # new: allow Chroma to serialize EF config
    def get_config(self) -> Dict[str, Any]:
        return {
            "name": self.name(),
            "base_url": self.base_url,
            "model": self.model,
            "timeout": self.timeout,
        }

_CHROMA_COL = None

def build_vector_store():
    global _CHROMA_COL
    if _CHROMA_COL is not None:
        return _CHROMA_COL

    os.makedirs(CHROMA_DIR, exist_ok=True)
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    emb_fn = OllamaEmbeddingFunction(
        base_url=OLLAMA_BASE_URL,
        model="bge-m3:latest",
    )
    _CHROMA_COL = client.get_or_create_collection(
        COLLECTION_NAME,
        embedding_function=emb_fn,
    )
    return _CHROMA_COL

def read_jsonl(path: str) -> Iterable[tuple[str, Optional[Dict[str, Any]]]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
            except Exception:
                obj = None
            yield s, obj

# =========================
# === SPARSE (BM25) =======
# =========================
_BM25_CACHE: dict[str, Any] = {}
# Base stopwords dari scikit-learn (bahasa Inggris)
_BASE_STOPWORDS = set(ENGLISH_STOP_WORDS)

for w in ["name", "names"]:
    _BASE_STOPWORDS.discard(w)

# Tambahan stopwords domain + Bahasa Indonesia
_EXTRA_STOPWORDS = {
    # kata2 query umum / “filler”
    "show", "all", "list", "their", "that", "have", "no",
    "include", "includes", "including", "sort", "sorted", "sorting",
    "result", "results", "record", "records", "row", "rows",
    "display", "return", "get", "give", "find", "fetch", "please", "kindly",
    "alphabetical", "alphabetically", "ascending", "descending", "distinct",

    # Bahasa Indonesia dasar
    "yang", "dan", "atau", "ke", "dari", "di", "itu", "ini",
    "tampilkan", "daftar",
}

# FINAL stopwords: English (sklearn) + ekstra custom
_STOPWORDS = _BASE_STOPWORDS | _EXTRA_STOPWORDS

def _normalize_text(txt: str) -> str:
    txt = txt.lower()
    txt = re.sub(r"(\d)[,\.](\d{3})(\b|$)", r"\1\2", txt)
    txt = re.sub(r"(\w)'s\b", r"\1", txt)
    txt = txt.replace("'", "")
    txt = re.sub(r"[^a-z0-9_]+", " ", txt)
    return txt.strip()

def _tokenize_bm25_query(q: str) -> List[str]:
    """
    Normalisasi + tokenisasi untuk BM25:
    - lowercase + bersihkan non-alfanumerik
    - buang stopword
    - stem plural sederhana:
      * countries -> country
      * customers -> customer
      (hanya kalau panjang > 3 supaya 'as' / 'is' nggak ke-cut)
    """
    s = _normalize_text(q)
    raw_tokens = s.split()
    toks: List[str] = []

    for t in raw_tokens:
        if not t:
            continue

        # --- plural stemming ringan ---
        if len(t) > 4 and t.endswith("ies"):
            # countries -> country
            t = t[:-3] + "y"
        elif len(t) > 3 and t.endswith("s") and not t.endswith("ss"):
            # customers -> customer, orders -> order
            t = t[:-1]

        if t and t not in _STOPWORDS:
            toks.append(t)

    return toks

def _jsonl_iter_records(jsonl_paths: List[str]) -> Iterable[Dict[str, Any]]:
    for path in jsonl_paths:
        p = Path(path)
        if not p.exists():
            continue
        for _, obj in read_jsonl(path):
            if isinstance(obj, dict):
                yield obj

def build_sparse_query_from_question(q: str) -> str:
    """
    Build query string untuk BM25 dari pertanyaan user.
    Di sini bisa tambahkan hint manual (synonym → canonical).
    """
    toks = _tokenize_bm25_query(q)
    q_l = (q or "").lower()

    # Hint: "list price" biasanya maksudnya MSRP
    if "list price" in q_l and "msrp" not in toks:
        toks.append("msrp")

    return " ".join(toks)

def _meta_table(md: Dict[str, Any]) -> str:
    """
    Ambil nama table dari metadata (table / tableName) dalam bentuk trimmed.
    """
    return str(md.get("table") or md.get("tableName") or "").strip()

def _meta_column(md: Dict[str, Any]) -> str:
    """
    Ambil nama column dari metadata (column / columnName) dalam bentuk trimmed.
    """
    return str(md.get("column") or md.get("columnName") or "").strip()

def _doc_tokens_from_meta(meta: Dict[str, Any]) -> List[str]:
    table = _meta_table(meta).lower()
    column = _meta_column(meta).lower()
    desc = str(meta.get("description") or "").lower()

    # aliases/roles dari DataHub → parse dengan benar (string JSON / list)
    aliases = _parse_str_or_list_lower(meta.get("tags_alias") or meta.get("aliases") or [])
    roles   = _parse_str_or_list_lower(meta.get("tags_role")  or meta.get("roles")   or [])

    extra: List[str] = []
    has_semantic = bool(aliases or roles)

    # ---------- TABLE-LEVEL SYNONYMS ----------
    # productlines spesial: selalu dibantu, karena query natural user sering "product lines / kategori"
    if table == "productlines":
        extra += ["product", "products", "product line", "product lines", "line", "category", "kategori"]
    else:
        # tabel lain: cuma kalau DataHub nggak supply alias/role sama sekali
        if not has_semantic:
            if table == "products":
                extra += ["product", "products", "item", "items", "barang", "produk"]
            elif table == "customers":
                extra += ["customer", "customers", "pelanggan", "client", "account"]
            elif table == "orders":
                extra += ["order", "orders", "pesanan", "transaksi"]
            elif table == "payments":
                extra += ["payment", "payments", "pembayaran"]
            # boleh tambah lain kalau perlu, tapi keep it small

    # ---------- COLUMN-LEVEL SYNONYMS (bridging kuat) ----------
    if column == "msrp":
        extra += ["price", "list price", "retail price", "harga jual"]
    elif column == "priceeach":
        extra += ["unit price", "selling price", "harga satuan"]
    elif column == "quantityinstock":
        extra += ["stock", "stok", "qty", "quantity", "on hand"]
    elif column == "quantityordered":
        extra += ["qty", "quantity ordered", "jumlah pesanan"]
    elif column == "paymentdate":
        extra += ["payment date", "tanggal pembayaran"]
    elif column == "amount":
        extra += ["amount", "nominal", "jumlah pembayaran"]

    # helper khusus description
    if column == "htmldescription":
        extra += ["html", "description", "html description"]
    if column == "textdescription":
        extra += ["text", "description", "text description"]

    bag = " ".join([table, column, desc] + aliases + roles + extra)
    return _tokenize_bm25_query(bag)

def jsonl_sparse_search(q: str, jsonl_paths: List[str], max_hits: int = 20) -> List[str]:
    """
    Kembalikan daftar candidate IDs (field_urn/urn fallback f"{source}:{table}::{column}") berdasar BM25.
    Di-cache per (paths hash) agar cepat.
    """
    # cache key by files + mtime
    sig = []
    for p in jsonl_paths:
        try:
            st = os.stat(p)
            sig.append(f"{p}:{int(st.st_mtime)}:{int(st.st_size)}")
        except Exception:
            sig.append(p)
    cache_key = "||".join(sorted(sig))

    if cache_key not in _BM25_CACHE:
        tokenized_corpus: List[List[str]] = []
        ids: List[str] = []

        for path in jsonl_paths:
            p = Path(path)
            if not p.exists():
                continue
            srcname = p.name

            for line_no, (raw, obj) in enumerate(read_jsonl(path), 1):
                # ---- CASE 1: line bukan dict → coba list-of-dict ----
                if not isinstance(obj, dict):
                    try:
                        maybe = json.loads(raw)
                    except Exception:
                        maybe = None

                    if isinstance(maybe, list):
                        for i, item in enumerate(maybe):
                            if not isinstance(item, dict):
                                continue
                            table = _meta_table(item).lower()
                            column = _meta_column(item).lower()

                            doc_id_raw = (
                                item.get("field_urn")
                                or item.get("urn")
                                or f"{srcname}:{table}::{column}"
                            )
                            doc_id = str(doc_id_raw)
                            tokens = _doc_tokens_from_meta(item)
                            tokenized_corpus.append(tokens)
                            ids.append(doc_id)
                        continue  # lanjut line berikutnya
                    # kalau bukan dict & bukan list-of-dict → skip line
                    continue

                # ---- CASE 2: obj sudah dict normal ----
                table = _meta_table(obj).lower()
                column = _meta_column(obj).lower()

                doc_id_raw = (
                    obj.get("field_urn")
                    or obj.get("urn")
                    or f"{srcname}:{table}::{column}"
                )
                doc_id = str(doc_id_raw)

                tokens = _doc_tokens_from_meta(obj)
                tokenized_corpus.append(tokens)
                ids.append(doc_id)

        if not tokenized_corpus:
            _BM25_CACHE[cache_key] = {"bm25": None, "ids": []}
        else:
            _BM25_CACHE[cache_key] = {
                "bm25": BM25Okapi(tokenized_corpus),
                "ids": ids,
            }

    entry = _BM25_CACHE[cache_key]
    if not entry["bm25"]:
        return []

    query_tokens = _tokenize_bm25_query(q)
    scores = entry["bm25"].get_scores(query_tokens)

    # <<< FIX: paksa jadi list supaya 'not scores' nggak pakai numpy boolean >>>
    if not isinstance(scores, list):
        scores = list(scores)

    # NEW: kalau semua skor == 0 → anggap sparse "nggak tau apa-apa"
    if not scores or max(scores) <= 0:
        if SPARSE_DEBUG:
            print("[SPARSE] all scores are 0 → returning no sparse hits")
        return []

    ranked = sorted(zip(entry["ids"], scores), key=lambda x: x[1], reverse=True)
    out = [doc_id for doc_id, _ in ranked[:max_hits]]

    if SPARSE_DEBUG:
        print("[SPARSE] top hits:")
        for doc_id, sc in ranked[:min(max_hits, 10)]:
            print(f"  - {doc_id}  score={sc:.3f}")

    return out

def _doc_matches_sparse(md: Dict[str, Any], sparse_ids: List[str]) -> bool:
    if not md or not sparse_ids:
        return False

    sid_lower = {x.lower() for x in sparse_ids}

    urn = str(md.get("field_urn") or md.get("urn") or "").lower()
    if urn and urn in sid_lower:
        return True

    table = _meta_table(md).lower()
    column = _meta_column(md).lower()
    if table and column:
        suffix = f":{table}::{column}"
        return any(x.lower().endswith(suffix) for x in sparse_ids)

    return False

# =========================
# === DENSE PROBE ========
# =========================
def _mentions_id_like_for_probe(text: str) -> bool:
    """
    Versi khusus untuk probe:
    - Abaikan frasa 'number of' supaya tidak dianggap ID-like.
      Contoh: 'total number of orders' → bukan minta id.
    """
    t = (text or "").lower()
    # buang pola 'number of', 'total number of', dll
    t = re.sub(r"\b(total\s+)?number\s+of\b", "", t)
    return bool(re.search(r"\b(id|identifier|nomor|no|code|kode)\b", t))

def build_dynamic_probe(question: str, qtype: Optional[str] = None, max_tokens: int = 16) -> str:
    """
    Bangun dense probe untuk embedding query, tapi:
    - Tail dibuat SESINGKAT mungkin.
    - Token ekstra hanya yang benar-benar relevan dengan pertanyaan.
    - Tidak lagi hardcode list panjang 'id date status sum total amount revenue ...'.
    """
    q_raw = question or ""
    q = q_raw.lower()
    toks_q = q.split()

    # ---- fitur dasar ----
    has_metric_word = any(k in q for k in [
    "amount", "revenue", "sales", "omset", "price", "harga",
    "gross", "net"
    ])
    has_count_word = any(k in q for k in [
        "count", "berapa banyak", "how many"
    ])
    has_time_word = any(k in q for k in [
        "today", "yesterday", "last", "this", "bulan", "minggu", "tahun",
        "quarter", "month", "week", "year", "tanggal", "hari",
        "mingguan", "bulanan", "202", "201"
    ])

    # entity utama classicmodels
    mentions_customer = any(k in q for k in ["customer", "customers", "pelanggan"])
    mentions_product  = any(k in q for k in ["product", "products", "produk"])
    mentions_order    = any(k in q for k in ["order", "orders", "pesanan"])
    mentions_employee = any(k in q for k in ["employee", "employees", "karyawan"])

    # entity tambahan (tabel lain)
    mentions_productline = any(
        k in q
        for k in [
            "product line", "product lines",
            "lini produk", "line produk",
            "kategori", "category"
        ]
    )
    mentions_office = any(
        k in q
        for k in [
            "office", "offices", "kantor",
            "territory", "territories", "region", "regional"
        ]
    )
    mentions_payment = any(
        k in q
        for k in [
            "payment", "payments", "pembayaran",
            "check ", "cheque", "cheques"
        ]
    )
    mentions_orderdetails = any(
        k in q
        for k in [
            "order detail", "order details",
            "line item", "line items",
            "item pesanan", "detail pesanan"
        ]
    )

    id_like = _mentions_id_like_for_probe(q)

    # -------------------------
    # 1) Tentukan base tokens
    # -------------------------
    base: List[str] = []

    # a) Untuk lookup murni → jangan banyak tambahan
    if qtype == "lookup":
        # Biasanya cukup entity hints saja (di bawah)
        base = []

    # b) Untuk agregasi → tambahkan sedikit metric/time hints
    elif qtype in {"aggregate_by", "aggregate_all"}:
        if has_metric_word:
            base += ["amount", "revenue", "price"]
        if has_count_word:
            base += ["count"]
        if has_time_word:
            base += ["date", "year", "month"]

    else:
        # qtype other: minimal hints
        if has_time_word:
            base += ["date"]

    # -------------------------
    # 2) Entity-specific hints
    # -------------------------
    if mentions_customer:
        base += ["customer"]
        if not id_like:
            base += ["customerName"]
        else:
            base += ["customerNumber"]

    if mentions_product:
        base += ["product"]
        if not id_like:
            base += ["productName"]
        else:
            base += ["productCode"]

    if mentions_order:
        base += ["order", "orderNumber"]

    if mentions_employee:
        base += ["employee", "employeeNumber"]

    if mentions_productline:
        base += ["productLine"]

    if mentions_office:
        base += ["officeCode", "city", "country", "territory"]

    if mentions_payment:
        base += ["payments", "amount", "paymentDate"]

    if mentions_orderdetails:
        base += ["orderdetails", "quantityOrdered", "priceEach"]

    # -------------------------
    # 2b) Geo hints (lebih dinamis)
    # -------------------------
    mentions_country_word = any(k in q for k in ["country", "negara"])
    mentions_city_word    = any(k in q for k in ["city", "kota"])
    mentions_region_word  = any(k in q for k in ["region", "wilayah", "territory", "territories"])

    geo_tokens: List[str] = []

    # CASE A: query sebut "country" tapi tidak sebut "city" → fokus ke country saja
    if mentions_country_word and not mentions_city_word:
        geo_tokens.append("country")

    # CASE B: query sebut "city" tapi tidak sebut "country" → fokus ke city saja
    elif mentions_city_word and not mentions_country_word:
        geo_tokens.append("city")

    # CASE C: sebut dua-duanya → boleh tambahin dua-duanya
    elif mentions_country_word and mentions_city_word:
        geo_tokens += ["country", "city"]

    # CASE D: nggak sebut city/country, tapi konteks-nya offices / region
    elif mentions_office or mentions_region_word:
        # offices biasanya lebih nyambung ke city, tapi country juga masih relevan
        geo_tokens += ["city", "country"]

    # CASE E: query tentang customers + ada pola "from <negara>"
    # misal: "customers from Japan", "pelanggan dari USA"
    if mentions_customer:
        if re.search(r"\bfrom\s+[A-Z][A-Za-z]+\b", q_raw):  # very simple heuristic
            if "country" not in geo_tokens:
                geo_tokens.append("country")

    # Tambahkan ke base, tapi tetap hormati max_tokens di bawah
    base += geo_tokens

    # Name-specific hints (generic)
    if any(k in q for k in ["name", "nama"]):
        base += ["name"]

    # Super pendek → beri sedikit “bumbu” supaya embedding nggak terlalu kosong
    if len(toks_q) <= 2 and not base:
        base += ["amount", "date"]

    # -------------------------
    # 3) Dedup + truncate
    # -------------------------
    seen = set()
    tokens: List[str] = []
    for t in base:
        t_norm = t.strip()
        if not t_norm:
            continue
        if t_norm not in seen:
            seen.add(t_norm)
            tokens.append(t_norm)
        if len(tokens) >= max_tokens:
            break

    # Kalau akhirnya tidak ada token tambahan, pakai question apa adanya
    if not tokens:
        return q_raw.strip()

    probe_tail = " ".join(tokens).strip()
    return f"{q_raw.strip()} {probe_tail}".strip()

# =========================
# === INGEST (PURE CHROMA)
# =========================
COMPARE_DEBUG = bool(int(os.getenv("COMPARE_DEBUG", "0")))

# kunci yang sering berubah & memicu upsert palsu → kita buang
VOLATILE_KEYS = {
    "_source", "updated_at", "lastModified", "timestamp", "ts"
}

def _norm_str(s: Any) -> str:
    s = "" if s is None else str(s)
    # rapikan whitespace tanpa mengubah isi
    s = "\n".join(re.sub(r"[ \t]{2,}", " ", line.rstrip()) for line in s.splitlines())
    return s.strip()

def _norm_list(x: Any) -> List[str]:
    # list stabil: json->list bila string JSON, trim, dedup casefold, lalu sort
    if isinstance(x, str):
        try:
            j = json.loads(x)
            x = j if isinstance(j, list) else [x]
        except Exception:
            x = [x]
    elif not isinstance(x, list):
        x = [x]
    cleaned, seen = [], set()
    for v in x:
        s = _norm_str(v)
        k = s.casefold()
        if s and k not in seen:
            seen.add(k)
            cleaned.append(s)
    return sorted(cleaned, key=str.casefold)

def _stable_json(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

# --- NEW: koersi nilai meta agar perbandingan tipe-stabil ---
def _coerce_json_like(v: Any) -> Any:
    """
    - Jika v string yang tampak seperti JSON list/dict → parse.
    - List → normalisasi via _norm_list.
    - Dict → biarkan, tapi saat dump akan sort_keys=True.
    """
    if isinstance(v, str):
        s = v.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                v = json.loads(s)
            except Exception:
                return v
        else:
            return v
    if isinstance(v, list):
        return _norm_list(v)
    return v

def _sanitize_for_compare(meta: Dict[str, Any]) -> Dict[str, Any]:
    """
    Versi khusus untuk perbandingan: parse kembali string JSON agar
    list/dict dibandingkan sebagai struktur, bukan sebagai string mentah.
    Abaikan _canon_hash agar tidak circular-diff.
    """
    out: Dict[str, Any] = {}
    for k, v in (meta or {}).items():
        if k in VOLATILE_KEYS or k == "_canon_hash":  # <<< ADD ignore hash on compare
            continue
        vv = _coerce_json_like(v)
        out[k] = vv
    return out

def _sanitize_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    """
    Hanya simpan tipe primitif. list/dict → JSON string STABIL.
    Buang kunci volatile agar tidak memicu 'updated' palsu.
    """
    out: Dict[str, Any] = {}
    for k, v in (meta or {}).items():
        if k in VOLATILE_KEYS:
            continue
        if isinstance(v, (str, int, float, bool)) or v is None:
            out[k] = _norm_str(v) if isinstance(v, str) else v
        elif isinstance(v, list):
            out[k] = _stable_json(_norm_list(v))
        elif isinstance(v, dict):
            out[k] = _stable_json(v)
        else:
            out[k] = _norm_str(v)
    return out

def _canon_doc(s: str) -> str:
    # hilangkan trailing spaces per-baris agar stabil
    return "\n".join(line.rstrip() for line in (s or "").splitlines()).strip()

def _canon_meta(d: Dict[str, Any]) -> str:
    """
    Dump meta ke JSON STABIL setelah tipe dinormalkan agar list/dict vs string JSON
    tidak bikin beda palsu. Abaikan _canon_hash pada tahap compare.
    """
    try:
        normalized = _sanitize_for_compare(d or {})
        return _stable_json(normalized)
    except Exception:
        return str(d or {})

# <<< ADD: Hash kanonik untuk early-skip >>>
def _compute_canon_hash(doc: str, meta: Dict[str, Any]) -> str:
    """
    Hitung hash stabil dari dokumen & metadata kanonik.
    Metadata di-normalize via _canon_meta (yang mengabaikan _canon_hash).
    """
    a = _canon_doc(doc or "")
    b = _canon_meta(meta or {})
    payload = f"{a}\n{b}".encode("utf-8")
    return hashlib.sha1(payload).hexdigest()

def _print_meta_doc_diff(doc_id: str, old_doc: str, new_doc: str, old_meta: Dict[str, Any], new_meta: Dict[str, Any]):
    if not COMPARE_DEBUG:
        return
    a_doc, b_doc = _canon_doc(old_doc), _canon_doc(new_doc)
    a_meta, b_meta = _canon_meta(old_meta), _canon_meta(new_meta)
    if a_doc == b_doc and a_meta == b_meta:
        return
    print(f"[DIFF] {doc_id}")
    if a_doc != b_doc:
        print("  - documents differ")
    if a_meta != b_meta:
        try:
            om = json.loads(a_meta); nm = json.loads(b_meta)
            keys = sorted(set(om.keys()) | set(nm.keys()))
            for k in keys:
                if om.get(k) != nm.get(k):
                    print(f"  * meta[{k}]: {om.get(k)!r} -> {nm.get(k)!r}")
        except Exception:
            print("  - metadatas differ (unparseable)")

def _ingest_one_json(col, meta: dict, source_name: str, debug_id: str):
    """
    Upsert satu item ke koleksi Chroma.
    Return: (added, updated, skipped)
    """
    # 1) tentukan id
    doc_id = meta.get("field_urn") or meta.get("urn")
    if not doc_id:
        table  = _norm_str(_meta_table(meta))
        column = _norm_str(_meta_column(meta))

        if not table and not column:
            return 0, 0, 1  # skip kalau minim info
        doc_id = f"{Path(source_name).name}:{table}::{column}".lower()

    # 2) normalisasi nilai penting
    aliases = _norm_list(meta.get("tags_alias") or meta.get("aliases") or [])
    roles   = _norm_list(meta.get("tags_role")  or meta.get("roles")   or [])
    table   = _norm_str(_meta_table(meta))
    column  = _norm_str(_meta_column(meta))
    desc    = _norm_str(meta.get("description") or "")

    # 3) dokumen teks STABIL
    new_doc = (
        f"table: {table}\n"
        f"column: {column}\n"
        f"description: {desc}\n"
        f"aliases: {', '.join(aliases)}\n"
        f"roles: {', '.join(roles)}"
    )

    # 4) metadata STABIL (tanpa _source & co), isi kolom kanonik
    meta2 = dict(meta or {})
    meta2["table"]       = table
    meta2["tableName"]   = table or meta2.get("tableName")
    meta2["column"]      = column
    meta2["columnName"]  = column or meta2.get("columnName")
    meta2["description"] = desc
    meta2["tags_alias"]  = aliases
    meta2["aliases"]     = aliases
    meta2["tags_role"]   = roles
    meta2["roles"]       = roles
    # JANGAN menyimpan _source agar tidak flip-flop
    if "_source" in meta2:
        del meta2["_source"]
    new_meta = _sanitize_metadata(meta2)

    # <<< ADD: hitung & tanam _canon_hash supaya run berikutnya bisa early-skip >>>
    canon_hash = _compute_canon_hash(new_doc, new_meta)
    new_meta["_canon_hash"] = canon_hash

    # 5) cek eksistensi
    try:
        got = col.get(ids=[doc_id], include=["documents", "metadatas"])
        exists = bool(got.get("ids"))
    except Exception:
        exists = False

    if not exists:
        col.add(ids=[doc_id], documents=[new_doc], metadatas=[new_meta])
        return 1, 0, 0

    # 6) bandingkan STABIL (tipe-agnostik) + early-skip

    # ambil dok & meta lama dari Chroma
    old_doc  = ((got.get("documents") or [""])[0]) if got else ""
    old_meta = ((got.get("metadatas") or [{}])[0]) if got else {}

    # --- EARLY 1: kalau last_ingested / lastIngested sama persis, langsung skip ---
    new_last = meta.get("last_ingested") or meta.get("lastIngested")
    old_last = old_meta.get("last_ingested") or old_meta.get("lastIngested")
    if new_last and old_last and str(new_last) == str(old_last):
        # ingestion timestamp sama → asumsi snapshot sama → nggak usah update
        return 0, 0, 1

    # --- EARLY 2: jika hash sama, langsung skip ---
    old_hash = str(old_meta.get("_canon_hash") or "").strip()
    if old_hash and old_hash == canon_hash:
        return 0, 0, 1

    # cek diff kanonik (doc + meta) sebagai safety net
    same_doc  = _canon_doc(old_doc)   == _canon_doc(new_doc)
    same_meta = _canon_meta(old_meta) == _canon_meta(new_meta)

    if same_doc and same_meta:
        # kalau hash lama belum ada, update sekali untuk “menanam” hash
        if not old_hash:
            col.update(ids=[doc_id], documents=[new_doc], metadatas=[new_meta])
            return 0, 1, 0
        return 0, 0, 1  # unchanged → skip

    # 7) ada beda → update + (opsional) tampilkan diff
    _print_meta_doc_diff(doc_id, old_doc, new_doc, old_meta, new_meta)
    col.update(ids=[doc_id], documents=[new_doc], metadatas=[new_meta])
    return 0, 1, 0

def ingest_node(state: RAGState) -> RAGState:
    paths = state["jsonl_paths"]
    col = build_vector_store()

    added = updated = skipped = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"JSONL not found: {path}")
            continue

        for line_no, (raw, obj) in enumerate(read_jsonl(path), 1):
            if obj is None:
                try:
                    maybe = json.loads(raw)
                    if isinstance(maybe, list):
                        for i, item in enumerate(maybe):
                            if isinstance(item, dict):
                                _a, _u, _s = _ingest_one_json(
                                    col, item, p.name, f"{path}:{line_no}[{i}]"
                                )
                                added += _a
                                updated += _u
                                skipped += _s
                        continue
                except Exception:
                    pass
                print(f"[INGEST] skip {path}:{line_no} (not dict/list)")
                continue

            _a, _u, _s = _ingest_one_json(col, obj, p.name, f"{path}:{line_no}")
            added += _a
            updated += _u
            skipped += _s

    state["added"] = added
    state["updated"] = updated
    state["skipped"] = skipped
    # NEW: warm up BM25 index sekali berdasarkan jsonl_paths
    try:
        jsonl_sparse_search("warmup", state["jsonl_paths"], max_hits=0)
    except Exception as e:
        if SPARSE_DEBUG:
            print(f"[SPARSE] warmup failed (ignored): {e!r}")
    return state

# =========================
# === COLUMN PRIOR BOOST ==
# =========================
def apply_column_priority_boost(
    q_l: str,
    table: str,
    col_name: str,
    roles: List[str],
    aliases: List[str],
) -> float:
    bonus = 0.0
    if any(k in q_l for k in ["name", "nama"]):
        if any(r == "name" or "name" in r for r in roles) or "name" in col_name:
            bonus += NAME_QUERY_BONUS
    if table.lower() == "customers":
        mentions_customer = any(k in q_l for k in ["customer", "customers", "pelanggan"])
        asks_id_like = any(k in q_l for k in [" id", "id ", " id,", "identifier", "kode", "code", "customer number"])
        if mentions_customer and not asks_id_like:
            if "customername" in col_name or "customer_name" in col_name:
                bonus += 0.6
            if "customernumber" in col_name or col_name.endswith("number"):
                bonus -= 0.35
    if table.lower() == "products":
        mentions_product = any(k in q_l for k in ["product", "products", "produk"])
        asks_id_like = any(k in q_l for k in [" id", "id ", " id,", "identifier", "kode", "code", "product code"])
        # Kalau user ngomong produk tapi nggak eksplisit minta ID/kode,
        # dorong productName, "turunin" productCode sedikit.
        if mentions_product and not asks_id_like:
            if "productname" in col_name or "product_name" in col_name:
                bonus += 0.5   # boleh kamu atur naik/turun
            if "productcode" in col_name or col_name.endswith("code"):
                bonus -= 0.3   # penalti ringan supaya kalah dari nama
    if any(k in q_l for k in [" id ", "id ", " id,", "identifier", "kode", "code"]):
        if (
            any(r == "id" or "id" in r for r in roles)
            or "number" in col_name
            or col_name.lower().endswith("id")
        ):
            bonus += ID_QUERY_BONUS

    col_lower = col_name.lower()
    role_set = set(roles)
    alias_set = set(aliases)
    for rule in COLUMN_PRIORITY_CONFIG:
        tables = [t.lower() for t in rule.get("tables", []) or []]
        if tables and table.lower() not in tables:
            continue

        role_any = rule.get("role_any") or []
        if role_any and not any(r in role_set for r in role_any):
            continue

        alias_any = rule.get("alias_any") or []
        if alias_any and not any(a in alias_set for a in alias_any):
            continue

        column_contains = rule.get("column_contains") or []
        if column_contains and not any(sub in col_lower for sub in column_contains):
            continue

        qt_any = rule.get("query_terms_any") or []
        if qt_any and not any(t in q_l for t in qt_any):
            continue

        qt_also = rule.get("query_terms_also_any") or []
        if qt_also and not any(t in q_l for t in qt_also):
            continue

        bonus += float(rule.get("boost", 0.0))

    return bonus

# ============================================================
# === NEW: simple query classifier + table scorer ============
# ============================================================
def _wants_name_only(question: str, entity_words: List[str]) -> bool:
    """
    Generic helper: True kalau user ngomongin entity (customer/product/...) 
    tapi tidak eksplisit minta id/number/code.
    """
    q = (question or "").lower()
    # buang "number of" supaya tidak dianggap id-like
    q = re.sub(r"\b(total\s+)?number\s+of\b", "", q)

    mention_entity = any(w in q for w in entity_words)
    mention_id_like = (
        re.search(r"\b(id|number|nomor|no|code|kode)\b", q)
        or any(f"{w} code" in q for w in entity_words)
    )
    return mention_entity and not mention_id_like


def _wants_customer_name_only(question: str) -> bool:
    # pelanggan/customer tapi tidak minta id/number/code
    return _wants_name_only(question, ["customer", "customers", "pelanggan"])


def _wants_product_name_only(question: str) -> bool:
    # produk/product tapi tidak minta id/number/code
    return _wants_name_only(question, ["product", "products", "produk"])

def _is_productline_lookup(question: str) -> bool:
    """
    True kalau user lagi minta daftar product lines (bukan agregat).
    Contoh:
      - "show all product lines and their descriptions"
      - "list all product lines"
    Dipakai bareng qtype == 'lookup' supaya nggak mengganggu query agregasi.
    """
    q = (question or "").lower()

    # fokus ke frasa product line(s)
    if "product line" not in q and "product lines" not in q and "lini produk" not in q and "line produk" not in q:
        return False

    # kalau ada kata-kata agregasi → biar classifier saja yang urus, jangan paksa single table
    if any(kw in q for kw in ["total", "sum", "avg", "average", "mean", "count", "berapa banyak", "how many"]):
        return False

    return True

def _is_simple_orders_lookup(question: str) -> bool:
    """
    True kalau user cuma minta daftar order (misal orderNumber) 
    tanpa ngomong 'order details' / 'line items' dan tanpa agregasi.
    Contoh:
      - "show all order numbers between ..."
      - "list order numbers for 2004"
    """
    q = (question or "").lower()

    # harus ada frasa 'order number(s)' biar ini spesifik
    if "order number" not in q and "order numbers" not in q:
        return False

    # kalau ngomong detail/line items → biar classifier & join yang urus
    if any(kw in q for kw in ["order detail", "order details", "line item", "line items"]):
        return False

    # kalau ada kata agregasi → jangan paksa single-table
    if any(
        kw in q
        for kw in ["total", "sum", "avg", "average", "mean", "count",
                   "berapa banyak", "how many", "jumlah "]
    ):
        return False

    return True
    
def classify_query(q: str) -> str:
    """
    return one of: 'lookup', 'aggregate_by', 'aggregate_all', 'other'
    """
    ql = (q or "").lower().strip()

    # ---------- 0) RULE KHUSUS: customers + total number of orders per customer ----------
    if (
        "customers" in ql
        and "total" in ql
        and "number of orders" in ql
    ):
        return "aggregate_by"

    # ---------- (A) buang kata2 sorting dulu, supaya 'by' di sini tidak disalah-artikan ----------
    qg = (
        ql.replace("sort by", "")
           .replace("sorted by", "")
           .replace("order by", "")
    )

    # ---------- 1) aggregate_by: 'group by', 'per X', atau 'total ... by ...' ----------
    if "group by" in qg or " per " in qg:
        return "aggregate_by"

    if re.search(r"\bby\b", qg) and any(
        kw in qg
        for kw in ["total", "sum", "avg", "average", "mean", "count", "jumlah", "how many", "berapa banyak"]
    ):
        # contoh: "total sales by country", "avg payment amount by customer"
        return "aggregate_by"

    # ---------- 2) aggregate_all: hanya mau 1 angka / 1 nilai summary ----------
    if (
        "how many" in ql
        or "berapa banyak" in ql
        or "jumlah " in ql
        or re.search(r"\bcount\b", ql)
    ):
        return "aggregate_all"

    if any(kw in ql for kw in ["sum", "total", "avg", "average", "mean"]):
        return "aggregate_all"

    # ---------- 3) DISTINCT ----------
    if "distinct" in ql:
        # kalau ada rasa "berapa/count" → aggregate_all
        if (
            "how many" in ql
            or "berapa" in ql
            or "jumlah" in ql
            or re.search(r"\bcount\b", ql)
        ):
            return "aggregate_all"
        # kalau cuma "show/list distinct ..." → lookup dimensi
        return "lookup"

    # ---------- 4) pola listing → lookup ----------
    if any(kw in ql for kw in ["list", "show", "tampilkan", "daftar", "all"]):
        return "lookup"

    # ---------- 5) very short entity mention → lookup ----------
    toks = ql.split()
    if len(toks) <= 4 and any(t in ql for t in ["customer", "pelanggan", "product", "order", "employee"]):
        return "lookup"

    # ---------- fallback ----------
    return "other"

def _get_qtype(state: RAGState) -> str:
    """
    Ambil qtype dari state kalau sudah ada,
    kalau belum, pakai classify_query(question) lalu simpan ke state.
    """
    qtype = state.get("qtype")
    if qtype:
        return qtype
    q = state.get("question", "")
    qtype = classify_query(q)
    state["qtype"] = qtype
    return qtype

def _get_requires_regex(state: RAGState) -> bool:
    """
    Hitung satu kali apakah pertanyaan butuh REGEXP,
    lalu simpan ke state['requires_regex'].
    """
    val = state.get("requires_regex")
    if isinstance(val, bool):
        return val  # True atau False → langsung pakai

    q = state.get("question", "")
    needs = _question_requires_regex(q)
    state["requires_regex"] = needs
    return needs

def score_tables_from_reranked(
    reranked: List[Dict[str, Any]],
    query_tokens: List[str],
) -> Dict[str, float]:
    scores: Dict[str, float] = defaultdict(float)
    qt = [t.lower() for t in query_tokens]
    q_l = " ".join(qt)

    # entity hints dari token
    mentions_customer = ("customer" in q_l) or ("customers" in q_l) or ("pelanggan" in q_l)
    mentions_product  = ("product" in q_l) or ("products" in q_l) or ("produk" in q_l)
    mentions_order    = "order" in q_l or "orders" in q_l
    mentions_payment  = ("payment" in q_l) or ("payments" in q_l) or ("pembayaran" in q_l)

    # tambahan sesuai tabel classicmodels:
    mentions_office = any(
        k in q_l
        for k in ["office", "offices", "kantor", "territory", "territories", "region", "regional"]
    )
    mentions_productline = any(
        k in q_l
        for k in ["product line", "product lines", "kategori", "category", "lini produk", "line produk"]
    )
    mentions_orderdetails = any(
        k in q_l
        for k in ["order detail", "order details", "line item", "line items", "item pesanan", "detail pesanan"]
    )
    mentions_employee = any(
        k in q_l
        for k in ["employee", "employees", "karyawan", "sales rep", "salesrep"]
    )
    penalize_orderdetails_default = mentions_order and not mentions_orderdetails
    
    for it in reranked:
        md = it.get("meta") or {}
        tbl = _meta_table(md)
        if not tbl:
            continue
        desc = (md.get("description") or "").lower()
        base = 0.0

        tl = tbl.lower()

        # sinyal sparse
        if it.get("sparse"):
            base += 0.5

        # tabel disebut eksplisit di query
        if tl in q_l:
            base += 1.0

        # token ada di description
        for t in qt:
            if t and t in desc:
                base += 0.3

        # entity-specific boosts utama
        if mentions_customer and tl == "customers":
            base += 0.9
        if mentions_product and tl == "products":
            base += 0.9
        if mentions_order and tl == "orders":
            base += 0.7
        if mentions_payment and tl == "payments":
            base += 0.7

        # entity-specific boosts tambahan
        if mentions_office and tl == "offices":
            base += 0.8
        if mentions_productline and tl == "productlines":
            base += 0.8
        if mentions_orderdetails and tl == "orderdetails":
            base += 0.8
        if penalize_orderdetails_default and tl == "orderdetails":
            base -= 0.4   # angka ini bisa kamu tuning
        if mentions_employee and tl == "employees":
            base += 0.8

        scores[tbl] += base

    return scores

# =========================
# === RETRIEVE (PURE CHROMA)
# =========================
def _extract_aliases_for_map(md: Dict[str, Any]) -> List[str]:
    """
    Ambil aliases dari metadata (aliases/tags_alias/synonyms),
    pastikan bentuknya List[str] lowercase.
    """
    return _parse_str_or_list_lower(
        md.get("aliases") or md.get("tags_alias") or md.get("synonyms") or []
    )

def _parse_str_or_list_lower(x: Any) -> List[str]:
    """
    Normalisasi input yang bisa berupa:
    - list → jadikan list string lower-case
    - string JSON list → parse, lalu lower
    - string biasa → jadikan [string.lower()]
    - lainnya → []
    """
    if isinstance(x, str):
        s = x.strip()
        # coba parse JSON list kalau cocok
        try:
            j = json.loads(s)
            if isinstance(j, list):
                return [str(v).lower() for v in j]
            # kalau bukan list, fallback ke string itu sendiri
            return [s.lower()]
        except Exception:
            return [s.lower()]
    elif isinstance(x, list):
        return [str(v).lower() for v in x]
    elif x is None:
        return []
    else:
        return [str(x).lower()]

def _extract_roles_aliases(md: Dict[str, Any]) -> tuple[List[str], List[str]]:
    """
    Helper untuk ambil roles & aliases dalam bentuk list lowercase.
    Dipakai supaya nggak duplikasi pola get(tags_role/roles) & get(tags_alias/aliases).
    """
    roles = _parse_str_or_list_lower(md.get("tags_role") or md.get("roles") or [])
    aliases = _parse_str_or_list_lower(md.get("tags_alias") or md.get("aliases") or [])
    return roles, aliases

def _build_allowed_block(filtered_allowed: Dict[str, List[str]]) -> str:
    """
    Build blok ### ALLOWED IDENTIFIERS dari dict table -> list kolom.
    """
    lines = []
    for t, cols in filtered_allowed.items():
        uniq = sorted(set(cols))
        lines.append(f"- {t}: {', '.join(uniq)}")
    body = "\n".join(lines) if lines else "- (none)"
    return "### ALLOWED IDENTIFIERS\n" + body

def _build_cards(filtered_allowed: Dict[str, List[str]], reranked: List[Dict[str, Any]]) -> str:
    """
    Build kartu-kartu preview table/column untuk context LLM.
    Inject juga kolom penting yang belum muncul, misalnya customers.customerName.
    """
    cards: List[str] = []
    present = set()

    # kartu dari reranked
    for it in reranked:
        md = it.get("meta") or {}
        table = _meta_table(md)
        column = _meta_column(md)
        if not table or not column:
            continue
        if table not in filtered_allowed:
            continue

        desc = (md.get("description") or "").strip()
        if len(desc) > 160:
            desc = desc[:160] + "..."
        cards.append(f"- table: {table} | column: {column} | desc: {desc}")
        present.add((table.lower(), column.lower()))

    # inject yang belum muncul
    for tbl, cols in filtered_allowed.items():
        for col in cols:
            key = (tbl.lower(), col.lower())
            if key in present:
                continue
            short_desc = ""
            if tbl.lower() == "customers" and col.lower() == "customername":
                short_desc = "customer display name / company name"
            cards.append(f"- table: {tbl} | column: {col} | desc: {short_desc}")
            present.add(key)

    return "\n".join(cards)

def retrieve_node(state: RAGState) -> RAGState:
    col = build_vector_store()
    q = (state["question"] or "").strip()
    q_l = q.lower()
    qtype = _get_qtype(state)

    wants_customer_no_id = _wants_customer_name_only(q)
    productline_lookup = _is_productline_lookup(q)
    orders_simple_lookup = _is_simple_orders_lookup(q)

    jsonl_paths = state.get("jsonl_paths", [])

    # NEW: gunakan hasil sparse_query_clean untuk token debug,
    #      supaya SPARSE DEBUG betul-betul mencerminkan query yang dikirim ke BM25.
    sparse_query_clean = build_sparse_query_from_question(q)
    sparse_tokens = _tokenize_bm25_query(sparse_query_clean)


    sparse_field_urns = jsonl_sparse_search(sparse_query_clean, jsonl_paths, max_hits=SPARSE_MAX_HITS)

    # --- pretty print untuk hasil SPARSE ---
    sparse_docs_pretty: List[str] = []
    try:
        if sparse_field_urns:
            got = col.get(ids=sparse_field_urns, include=["metadatas"])
            ids   = (got.get("ids") or [])
            metas = (got.get("metadatas") or [])

            # Penting: bikin mapping id -> meta, lalu iterasi sesuai urutan sparse_field_urns
            meta_by_id = {i: (m or {}) for i, m in zip(ids, metas)}

            for doc_id in sparse_field_urns:
                md = meta_by_id.get(doc_id, {}) or {}
                tbl = _meta_table(md) or "?"
                c   = _meta_column(md) or "?"
                sparse_docs_pretty.append(f"table={tbl}  column={c}")
    except Exception as e:
        # fallback: kalau gagal ambil meta dari Chroma, tampilkan ID-nya saja
        sparse_docs_pretty = list(sparse_field_urns or [])
        if RETRIEVE_DEBUG:
            print(f"[WARN] Sparse meta fetch failed: {e!r}")
            
    sparse_has_customername = any(
        u.lower().endswith("::customername") and ".customers," in u.lower()
        for u in sparse_field_urns
    )

    base_k = max(12, TOP_K)
    dense_k = base_k
    if wants_customer_no_id and not sparse_has_customername:
        dense_k = max(28, base_k)

    query_text = build_dynamic_probe(q, qtype=qtype)
    res = col.query(
        query_texts=[query_text],
        n_results=dense_k,
        include=["documents", "metadatas", "distances"],
    )

    dense_docs   = (res.get("documents")  or [[]])[0]
    dense_metas  = (res.get("metadatas")  or [[]])[0]
    dense_dists  = (res.get("distances")  or [[]])[0]
    dense_ids    = (res.get("ids")        or [[]])[0]

    n = min(
        len(dense_docs),
        len(dense_metas),
        len(dense_dists) if dense_dists else len(dense_docs),
        len(dense_ids) if dense_ids else len(dense_docs),
    )
    dense_docs  = dense_docs[:n]
    dense_metas = dense_metas[:n]
    dense_dists = (dense_dists or [0.0] * n)[:n]
    dense_ids   = (dense_ids or [""] * n)[:n]

    dense_has_customername = any(
        (_meta_table(md).lower() == "customers")
        and (_meta_column(md).lower() in ("customername", "customer_name"))
        for md in dense_metas
    )

    scored = []
    for rank, (doc_text, md, dist, _id) in enumerate(zip(dense_docs, dense_metas, dense_dists, dense_ids)):
        base = -float(dist) if dist is not None else 0.0
        sparse_hit = _doc_matches_sparse(md, sparse_field_urns)

        table_name = _meta_table(md)
        col_name   = _meta_column(md).lower()

        # parse roles & aliases (DRY helper)
        roles, aliases = _extract_roles_aliases(md)

        boosted = base + (SPARSE_BONUS if sparse_hit else 0.0)
        boosted += apply_column_priority_boost(
            q_l=q_l,
            table=table_name,
            col_name=col_name,
            roles=roles,
            aliases=aliases,
        )

        scored.append({
            "rank": rank,
            "id": _id,
            "doc": doc_text,
            "meta": md or {},
            "dist": float(dist) if dist is not None else 0.0,
            "base": base,
            "sparse": sparse_hit,
            "boosted": boosted,
        })

    # auto-inject sangat terbatas
    if wants_customer_no_id and sparse_has_customername and not dense_has_customername:
        sparse_customername_urn = None
        for u in sparse_field_urns:
            if u.lower().endswith("::customername"):
                sparse_customername_urn = u
                break
        if sparse_customername_urn:
            present_ids = {item["id"] for item in scored if item.get("id")}
            if sparse_customername_urn not in present_ids:
                got = col.get(ids=[sparse_customername_urn])
                if got and got.get("ids"):
                    md  = (got.get("metadatas") or [{}])[0]
                    doc = (got.get("documents") or [""])[0]

                    roles, aliases = _extract_roles_aliases(md)

                    table_name = _meta_table(md)
                    col_name   = _meta_column(md).lower()

                    max_boost = max((x["boosted"] for x in scored), default=0.0)
                    boosted = max_boost + 0.05
                    boosted += apply_column_priority_boost(
                        q_l=q_l,
                        table=table_name,
                        col_name=col_name,
                        roles=roles,
                        aliases=aliases,
                    )

                    scored.append({
                        "rank": len(scored),
                        "id": sparse_customername_urn,
                        "doc": doc,
                        "meta": md or {},
                        "dist": 0.0,
                        "base": 0.0,
                        "sparse": True,
                        "boosted": boosted,
                    })

    reranked = sorted(scored, key=lambda x: x["boosted"], reverse=True)
    reranked = reranked[:MAX_CTX_DOCS]

    if RETRIEVE_DEBUG:
        line = _print_banner("RETRIEVE DEBUG")

        print(f"QUESTION      : {q}")
        print(f"DENSE QUERY   : {query_text}")
        print(f"SPARSE QUERY  : {sparse_query_clean}")
        print(f"TOKENS        : {sparse_tokens}")
        print(f"QTYPE         : {qtype}\n")

        # =========================
        # SPARSE CANDIDATES
        # =========================
        total_sparse = len(sparse_docs_pretty)
        max_show = min(7, total_sparse)
        print(f"SPARSE CANDIDATES (top {max_show} of {total_sparse}):")
        if total_sparse == 0:
            print("  <none>\n")
        else:
            # prefix row: "  [01] " = 7 chars → header juga 7 spasi
            print(f"{'':7}{'table':<14} {'column':<30}")
            for i, s in enumerate(sparse_docs_pretty[:max_show], start=1):
                # s format: "table=xxx  column=yyy"
                try:
                    _, rest = s.split("table=", 1)
                    table_part, col_part = rest.split("column=", 2)
                    table = table_part.strip() or "?"
                    column = col_part.strip() or "?"
                except Exception:
                    table = "?"
                    column = s
                print(f"  [{i:02d}] {table:<14} {column:<30}")
            if total_sparse > max_show:
                remaining = total_sparse - max_show
                print(f"  ... (+{remaining} more)")
            print()

        # =========================
        # DENSE CANDIDATES
        # =========================
        total_dense = len(dense_metas)
        max_show = min(7, total_dense)
        print(f"DENSE CANDIDATES (top {max_show} of {total_dense}):")
        if total_dense == 0:
            print("  <none>\n")
        else:
            print(f"{'':7}{'table':<14} {'column':<30}")
            for i in range(max_show):
                md  = dense_metas[i] or {}
                tbl = _meta_table(md) or "?"
                col = _meta_column(md) or "?"
                print(f"  [{i+1:02d}] {tbl:<14} {col:<30}")
            if total_dense > max_show:
                remaining = total_dense - max_show
                print(f"  ... (+{remaining} more)")
            print()

        # =========================
        # RERANKED RESULT
        # =========================
        total_rerank = len(reranked)
        max_show = min(7, total_rerank)
        print(f"RERANKED RESULT (top {max_show} of {total_rerank}):")
        if total_rerank == 0:
            print("  <no reranked candidates>")
        else:
            print(f"{'':7}{'table':<14} {'column':<30} {'sparse':>8}")
            for i, item in enumerate(reranked[:max_show], start=1):
                md   = item.get("meta") or {}
                tbl  = _meta_table(md) or "?"
                col  = _meta_column(md) or "?"
                sparse_flag = "✓" if item.get("sparse") else "✗"
                print(f"  [{i:02d}] {tbl:<14} {col:<30} {sparse_flag:>8}")
            if total_rerank > max_show:
                remaining = total_rerank - max_show
                print(f"  ... (+{remaining} more)")
        print(f"{line}\n")

    # build allowed + alias_map
    allowed   = defaultdict(list)
    alias_map = defaultdict(dict)
    for item in reranked:
        md = item["meta"] or {}
        t  = _meta_table(md)
        c  = _meta_column(md)
        if t and c:
            allowed[t].append(c)
    
            aliases2 = _extract_aliases_for_map(md)
            for al in aliases2:
                al = al.strip()
                if al:
                    alias_map[t][al] = c   # al sudah lowercase dari helper

    token_for_score = _tokenize_bm25_query(q)
    table_scores = score_tables_from_reranked(reranked, token_for_score)

    if qtype == "lookup" and wants_customer_no_id:
        filtered_allowed = {t: cols for t, cols in allowed.items() if t.lower() == "customers"}
    elif qtype == "lookup" and productline_lookup:
        # khusus: lookup product lines → pakai 1 tabel saja
        only_pl = {t: cols for t, cols in allowed.items() if t.lower() == "productlines"}
        filtered_allowed = only_pl if only_pl else dict(allowed)
    elif qtype == "lookup" and orders_simple_lookup:
        # khusus: simple "order numbers" lookup → fokus ke tabel orders saja
        only_orders = {t: cols for t, cols in allowed.items() if t.lower() == "orders"}
        filtered_allowed = only_orders if only_orders else dict(allowed)
    else:
        if table_scores:
            sorted_tbls = sorted(table_scores.items(), key=lambda x: x[1], reverse=True)
            top_tables = {name for name, score in sorted_tbls[:2] if score > 0.0}
            if top_tables:
                filtered_allowed = {t: cols for t, cols in allowed.items() if t in top_tables}
            else:
                filtered_allowed = dict(allowed)
        else:
            filtered_allowed = dict(allowed)

    if wants_customer_no_id:
        for t_key in list(filtered_allowed.keys()):
            if t_key.lower() == "customers":
                if "customerName" not in filtered_allowed[t_key]:
                    filtered_allowed[t_key].append("customerName")

    allowed_block = _build_allowed_block(filtered_allowed)
    cards_text    = _build_cards(filtered_allowed, reranked)

    context = (allowed_block + "\n\n" + cards_text).strip()
    context += f"\n\n### _INTERNAL_HINT\n{json.dumps({'alias_map': alias_map}, ensure_ascii=False)}"

    state["context"] = context
    return state

# =========================
# === SQL POSTPROCESS =====
# =========================
_JOIN_KW = r"(?:inner\s+join|left\s+join|right\s+join|full\s+join|join)"

def _has_join(sql: str) -> bool:
    return re.search(rf"\b{_JOIN_KW}\b", sql, flags=re.I) is not None

def _should_use_backtick(identifier: str) -> bool:
    if not identifier: return False
    clean = identifier.strip('`"')
    if not clean: return False
    first_char = clean[0]
    if first_char.isupper() or first_char.isdigit():
        return True
    if not re.match(r'^[a-z_][a-z0-9_]*$', clean, re.I):
        return True
    return False

def _smart_backtick(identifier: str) -> str:
    clean = identifier.strip('`"')
    return f"`{clean}`" if _should_use_backtick(clean) else clean

def _split_qualified(name: str) -> List[str]:
    return [p.strip('`" ') for p in name.split(".") if p.strip("\"` ")]

def _fix_bad_backticks(sql: str) -> str:
    def repl_fqn(m: re.Match) -> str:
        inside = m.group(1)
        if "." not in inside:
            return _smart_backtick(inside)
        parts = _split_qualified(inside)
        if len(parts) == 0:
            return inside
        return ".".join(_smart_backtick(p) for p in parts)
    sql = re.sub(r"`([^`]+?\.[^`]+?)`", repl_fqn, sql)
    sql = re.sub(r"(\b[A-Za-z_][\w]*)\.`([^`]+?)`",
                 lambda m: f"{_smart_backtick(m.group(1))}."
                           + ".".join(_smart_backtick(p) for p in _split_qualified(m.group(2))), sql)
    sql = re.sub(r"`([^`]+?)`\.(\b[A-Za-z_][\w]*)",
                 lambda m: ".".join(_smart_backtick(p) for p in _split_qualified(m.group(1)))
                           + f".{_smart_backtick(m.group(2))}", sql)
    sql = re.sub(r'`([^`]+?)`(?!\s*\.)', lambda m: _smart_backtick(m.group(1)), sql, flags=re.I)
    return sql

def _explicit_limit_from_question(question: str) -> Optional[int]:
    q = (question or "").lower()
    m = re.search(r"\blimit\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\btop\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\b(show|first|tampilkan|ambil)\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(2)), 1_000_000))
    return None

def _maybe_add_limit(question: str, sql: str) -> str:
    if re.search(r"\blimit\s+\d+\b", sql, flags=re.I):
        return sql
    n = _explicit_limit_from_question(question)
    if n is None:
        return sql
    s = sql.rstrip().rstrip(";")
    return s + f" LIMIT {n};"

def _wants_distinct_from_question(question: str) -> bool:
    q = (question or "").lower()
    hard_signals = [
        "distinct",
        "no duplicate",
        "without duplicates",
        "tanpa duplikat",
        "tanpa duplicate",
        "hanya yang unik",
        "dedup",
        "de-dup",
    ]
    if any(s in q for s in hard_signals):
        return True
    if "unique" in q and ("rows" in q or "records" in q or "result" in q):
        return True
    return False

def _has_distinct(sql: str) -> bool:
    return bool(re.search(r"^\s*select\s+distinct\b", sql, flags=re.I))

def _add_distinct(sql: str) -> str:
    if _has_distinct(sql):
        return sql
    return re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)

def _remove_distinct(sql: str) -> str:
    return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)

def _ensure_reasonable_distinct(sql: str, question: str) -> str:
    if not sql:
        return sql
    if _wants_distinct_from_question(question):
        return _add_distinct(sql)
    return _remove_distinct(sql)

def _fix_missing_space_before_clauses(sql: str) -> str:
    sql = re.sub(r"\s+(?=(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b)", " ", sql, flags=re.I)
    sql = re.sub(
        r"([)\],;`])(?=(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b)",
        r"\1 ",
        sql,
        flags=re.I,
    )
    sql = re.sub(r"([A-Za-z0-9_\.])(?=(LIMIT|WHERE|HAVING)\b)", r"\1 ", sql)
    sql = re.sub(r"([A-Za-z0-9_\.])(?=ORDER\s+BY\b)", r"\1 ", sql)
    sql = re.sub(r"([A-Za-z0-9_\.])(?=GROUP\s+BY\b)", r"\1 ", sql)
    sql = re.sub(r"\s+(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b", r" \1", sql, flags=re.I)
    return sql

def _collapse_spaces_outside_quotes(s: str) -> str:
    parts = re.split(r"('(?:''|[^'])*')", s)
    for i in range(len(parts)):
        if i % 2 == 0:
            parts[i] = re.sub(r"[ \t]{2,}", " ", parts[i])
    return "".join(parts)

def _extract_order_by_clause(sql: str) -> Optional[re.Match]:
    return re.search(r"\border\s+by\s+(.*?)(?=(\blimit\b|;|$))", sql, flags=re.I | re.S)

def _remove_order_by_clause(sql: str) -> str:
    return re.sub(r"\s+order\s+by\s+.*?(?=(\blimit\b|;|$))", "", sql, flags=re.I | re.S)

def _question_requests_ordering(question: str) -> bool:
    q = (question or "").lower()
    keywords = [
        "order by", "sort", "sorted", "ascending", "descending", "asc", "desc",
        "alphabetical", "alphabetically", "latest", "newest", "oldest",
        "highest", "lowest", "top", "bottom", "most expensive", "cheapest",
        "urut", "urutkan", "diurutkan", "menaik", "menurun",
        "terbesar", "terkecil", "tertinggi", "terendah", "terbaru", "terlama",
        "termahal", "termurah",
    ]
    return any(k in q for k in keywords)

def _strip_unneeded_count_distinct(sql: str) -> str:
    low = sql.lower()
    if "from customers" in low and "join orders" in low:
        sql = re.sub(
            r"COUNT\s*\(\s*DISTINCT\s+o\.orderNumber\s*\)",
            "COUNT(o.orderNumber)",
            sql,
            flags=re.IGNORECASE,
        )
    return sql
    
def _prefer_name_over_id_generic(
    sql: str,
    question: str,
    wants_name_fn,
    name_column: str,
    id_columns: List[str],
) -> str:
    """
    Ganti penggunaan kolom ID menjadi kolom nama yang lebih 'friendly'
    untuk SELECT & GROUP BY, kalau pertanyaan memang prefer 'name'.
    Contoh:
      - customerNumber -> customerName
      - productCode    -> productName
    """
    if not wants_name_fn(question):
        return sql

    # contoh: "customerNumber|customer_id|..."
    id_pattern = "|".join(re.escape(c) for c in id_columns)

    # CASE 1: SELECT alias.idCol        → SELECT alias.nameCol AS nameCol
    sql = re.sub(
        rf"SELECT\s+([A-Za-z_][A-Za-z0-9_]*)\.({id_pattern})(\s+AS\s+\2)?",
        rf"SELECT \1.{name_column} AS {name_column}",
        sql,
        flags=re.I,
    )

    # CASE 2: SELECT alias.idCol, ...   → SELECT alias.nameCol, ...
    sql = re.sub(
        rf"SELECT\s+([A-Za-z_][A-Za-z0-9_]*)\.({id_pattern})\s*,",
        rf"SELECT \1.{name_column},",
        sql,
        flags=re.I,
    )

    # CASE 3: GROUP BY alias.idCol      → GROUP BY alias.nameCol
    sql = re.sub(
        rf"GROUP BY\s+([A-Za-z_][A-Za-z0-9_]*)\.({id_pattern})\b",
        rf"GROUP BY \1.{name_column}",
        sql,
        flags=re.I,
    )

    return sql


def _prefer_friendly_names(sql: str, question: str) -> str:
    """
    DRY: apply rule yang sama untuk beberapa entity sekaligus.
    - customers: customerNumber -> customerName
    - products : productCode    -> productName
    (hanya kalau user tidak minta id/number/code)
    """
    # customers
    sql = _prefer_name_over_id_generic(
        sql=sql,
        question=question,
        wants_name_fn=_wants_customer_name_only,
        name_column="customerName",
        id_columns=["customerNumber"],
    )

    # products
    sql = _prefer_name_over_id_generic(
        sql=sql,
        question=question,
        wants_name_fn=_wants_product_name_only,
        name_column="productName",
        id_columns=["productCode"],
    )

    return sql
    
def _dynamic_prune_joins_if_single_table_possible(sql: str, question: str, ctx: str = "") -> str:
    """
    Coba hilangkan JOIN yang tidak perlu secara dinamis:
    - Lihat alias -> table dari FROM/JOIN
    - Lihat kolom yang dipakai per alias
    - Lihat schema dari ALLOWED IDENTIFIERS
    - Kalau semua kolom yang dipakai lewat alias lain juga ada di 1 tabel anchor,
      alias itu dianggap redundant dan JOIN-nya bisa dihapus.

    Hanya aktif untuk query jenis 'lookup' dan 'aggregate_all' (lebih aman).
    """
    if not sql or not _has_join(sql):
        return sql

    allowed = _parse_allowed_identifiers(ctx)
    if not allowed:
        return sql

    alias2table = _parse_from_join_aliases(sql)
    if len(alias2table) <= 1:
        return sql  # nggak ada join beneran

    usage = _collect_alias_column_usage(sql)

    # Pilih alias anchor: yang paling banyak dipakai kolomnya
    alias_counts = {a: len(cols) for a, cols in usage.items() if a in alias2table}
    if not alias_counts:
        return sql

    primary_alias = max(alias_counts.items(), key=lambda x: x[1])[0]
    primary_table = alias2table.get(primary_alias)
    if not primary_table or primary_table not in allowed:
        return sql

    # schema lower-case
    allowed_lower = {t.lower(): {c.lower() for c in cols} for t, cols in allowed.items()}
    primary_cols = allowed_lower.get(primary_table.lower())
    if not primary_cols:
        return sql

    redundant_aliases: List[tuple[str, str]] = []

    for alias, cols in usage.items():
        if alias == primary_alias:
            continue
        table = alias2table.get(alias)
        if not table:
            continue
        if table not in allowed:
            continue  # kita nggak tahu schema-nya, jangan diutak-atik

        # semua kolom yang dipakai via alias ini harus tersedia juga di tabel anchor
        if all(col.lower() in primary_cols for col in cols):
            redundant_aliases.append((alias, table))

    if not redundant_aliases:
        return sql

    new_sql = sql

    for alias, table in redundant_aliases:
        # 1) ganti alias.col -> primary_alias.col untuk kolom yang ada di tabel anchor
        pattern = re.compile(
            rf"\b{re.escape(alias)}\.\s*([A-Za-z_][A-Za-z0-9_]*)",
            flags=re.IGNORECASE,
        )

        def _repl(m: re.Match) -> str:
            col = m.group(1)
            if col.lower() in primary_cols:
                return f"{primary_alias}.{col}"
            return m.group(0)

        new_sql = pattern.sub(_repl, new_sql)

        # 2) hapus JOIN <table> <alias> ON ...
        join_pat = re.compile(
            rf"""
            \s+
            (?:INNER\s+|LEFT\s+|RIGHT\s+|FULL\s+OUTER\s+|FULL\s+)?JOIN
            \s+{re.escape(table)}\s+{re.escape(alias)}
            \s+ON
            \s+
            .*?
            (?=(\s+(?:INNER\s+|LEFT\s+|RIGHT\s+|FULL\s+OUTER\s+)?JOIN\b
                |\s+WHERE\b
                |\s+GROUP\s+BY\b
                |\s+ORDER\s+BY\b
                |\s+LIMIT\b
                |;|$))
            """,
            flags=re.IGNORECASE | re.VERBOSE | re.DOTALL,
        )
        new_sql = join_pat.sub(" ", new_sql, count=1)

    return new_sql

def _postprocess_sql(sql: str, question: str, context: str = "") -> str:
    if not sql:
        return sql
    s = sql.strip()

    def _norm_kw(text: str) -> str:
        CHUNKS = re.compile(
            r"(`[^`]*`|\"([^\"\\]|\\.)*\"|'([^'\\]|\\.)*')",
            re.DOTALL,
        )
        KEYWORDS = [
            "select","from","where","group by","order by","having","limit",
            "inner join","left join","right join","full join","join","on",
        ]

        def normalize_code_segment(seg: str) -> str:
            for kw in KEYWORDS:
                pattern = rf"(?<![A-Za-z0-9_]){kw}(?![A-Za-z0-9_])"
                seg = re.sub(pattern, lambda m: m.group(0).upper(), seg, flags=re.I)
            seg = re.sub(r"[ \t]+", " ", seg)
            return seg

        out, i = [], 0
        for m in CHUNKS.finditer(text):
            out.append(normalize_code_segment(text[i:m.start()]))
            out.append(m.group(0))
            i = m.end()
        out.append(normalize_code_segment(text[i:]))

        return "".join(out)

    s = _norm_kw(s)
    if not s.endswith(";"):
        s += ";"

    s = _fix_bad_backticks(s)
    s = _ensure_reasonable_distinct(s, question)
    s = _prefer_friendly_names(s, question)
    ob_match = _extract_order_by_clause(s)
    if ob_match and not _question_requests_ordering(question):
        s = _remove_order_by_clause(s)

    # NEW: dynamic join pruning
    s = _dynamic_prune_joins_if_single_table_possible(s, question, context)

    s = _maybe_add_limit(question, s)
    s = _fix_missing_space_before_clauses(s)
    s = _collapse_spaces_outside_quotes(s)
    s = _strip_unneeded_count_distinct(s)
    return s

def _normalize_sql_pipeline(raw_sql: str, question: str, ctx: str) -> str:
    """
    Pipeline standar untuk normalisasi SQL:
    - postprocess (normalize keyword, limit, distinct, dsb)
    - repair alias berdasarkan alias_map
    - postprocess lagi setelah alias diganti
    """
    if not raw_sql:
        return raw_sql
    s = _postprocess_sql(raw_sql, question, ctx)
    s = _repair_with_aliases(s, ctx)
    s = _postprocess_sql(s, question, ctx)
    return s

# ---------- Whitelist parsing + alias repair ----------
_ALNUM = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

def _extract_internal_hint(ctx: str) -> Dict[str, Dict[str, str]]:
    m = re.search(r"### _INTERNAL_HINT\s+({.*})\s*$", ctx, flags=re.S)
    if not m:
        return {}
    try:
        obj = json.loads(m.group(1))
        return obj.get("alias_map") or {}
    except Exception:
        return {}

def _parse_allowed_identifiers(ctx: str) -> Dict[str, set]:
    tables: Dict[str, set] = {}
    m = re.search(r"### ALLOWED IDENTIFIERS(.*?)(?:\n### |\Z)", ctx, flags=re.S)
    if not m:
        return tables
    block = m.group(1)
    for line in block.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue
        parts = line[2:].split(":", 1)
        if len(parts) != 2:
            continue
        table = parts[0].strip().strip("`")
        cols = [c.strip().strip("`") for c in parts[1].split(",") if c.strip()]
        if table:
            tables.setdefault(table, set()).update(cols)
    return tables

def _get_single_allowed_table(ctx: str) -> Optional[str]:
    """
    Jika ALLOWED IDENTIFIERS hanya berisi 1 tabel, return nama tabel itu.
    Kalau >1 atau 0 tabel, return None.
    """
    allowed = _parse_allowed_identifiers(ctx)
    if len(allowed) == 1:
        return next(iter(allowed.keys()))
    return None

def _extract_tables_from_sql(sql: str) -> set:
    tables = set()
    for m in re.finditer(r"\bfrom\s+([A-Za-z_][A-Za-z0-9_]*)", sql, flags=re.I):
        tables.add(m.group(1))
    for m in re.finditer(r"\bjoin\s+([A-Za-z_][A-Za-z0-9_]*)", sql, flags=re.I):
        tables.add(m.group(1))
    return tables

def _identify_used_identifiers(sql: str) -> Dict[Optional[str], set]:
    qualified = re.findall(
        r"`?([A-Za-z_][A-Za-z0-9_]*)`?\s*\.\s*`?([A-Za-z_][A-Za-z0-9_]*)`?",
        sql,
    )
    out: Dict[Optional[str], set] = {}
    for t, c in qualified:
        out.setdefault(t, set()).add(c)

    table_names = _extract_tables_from_sql(sql)
    for t in table_names:
        out.setdefault(t, set())

    keywords = {
        "select","from","where","group","by","order","limit","join","left","right",
        "inner","on","and","or","not","as","distinct","count","sum","avg","min","max",
        "having","asc","desc","like","in","between","is","null","case","when","then",
        "else","end"
    }

    for tok in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\b", sql):
        if tok.lower() in keywords:
            continue
        if tok in table_names:
            continue
        out.setdefault(None, set()).add(tok)

    return out

# --- NEW: parse alias FROM/JOIN ---
def _parse_from_join_aliases(sql: str) -> Dict[str, str]:
    """
    Ambil mapping alias -> base table dari klausa FROM dan JOIN.
    Contoh:
      FROM payments p
      JOIN customers c ON ...
    menghasilkan: {"p": "payments", "c": "customers"}
    """
    alias2table: Dict[str, str] = {}

    # FROM <table> [alias]
    for m in re.finditer(
        r"\bfrom\s+([A-Za-z_][A-Za-z0-9_]*)(?:\s+([A-Za-z_][A-Za-z0-9_]*))?",
        sql,
        flags=re.IGNORECASE,
    ):
        table = m.group(1)
        alias = m.group(2) or table
        alias2table[alias] = table

    # [INNER|LEFT|RIGHT|FULL] JOIN <table> [alias]
    for m in re.finditer(
        r"\b(?:inner\s+|left\s+|right\s+|full\s+outer\s+|full\s+)?join\s+([A-Za-z_][A-Za-z0-9_]*)(?:\s+([A-Za-z_][A-Za-z0-9_]*))?",
        sql,
        flags=re.IGNORECASE,
    ):
        table = m.group(1)
        alias = m.group(2) or table
        alias2table[alias] = table

    return alias2table


# --- NEW: collect penggunaan kolom per alias ---
def _collect_alias_column_usage(sql: str) -> Dict[str, set]:
    """
    Kembalikan mapping alias -> set kolom yang dipakai di query,
    berdasarkan pola alias.column (tidak bedakan posisi SELECT/WHERE/etc.).
    """
    usage: Dict[str, set] = defaultdict(set)
    for m in re.finditer(
        r"\b([A-Za-z_][A-Za-z0-9_]*)\s*\.\s*([A-Za-z_][A-Za-z0-9_]*)",
        sql,
        flags=re.IGNORECASE,
    ):
        alias, col = m.group(1), m.group(2)
        usage[alias].add(col)
    return usage


def _validate_identifiers(sql: str, ctx: str) -> List[str]:
    allowed = _parse_allowed_identifiers(ctx)
    used = _identify_used_identifiers(sql)
    errs: List[str] = []

    explicit_tables = {t for t in used.keys() if t is not None}

    single_table: Optional[str] = None
    if len(explicit_tables) == 1:
        single_table = next(iter(explicit_tables))
    elif len(explicit_tables) == 0 and len(allowed) == 1:
        single_table = next(iter(allowed.keys()))

    for tbl, cols in used.items():
        if tbl is None:
            continue
        if tbl not in allowed:
            errs.append(f"table `{tbl}` not in ALLOWED")
            continue
        bad = [c for c in cols if c not in allowed[tbl]]
        if bad:
            errs.append(f"{tbl} -> invalid columns: {', '.join(bad)}")

    if single_table:
        allowed_cols = allowed.get(single_table, set())
        for col in used.get(None, set()):
            if col not in allowed_cols:
                errs.append(f"{single_table} -> invalid column (unqualified): {col}")

    return errs

def _repair_with_aliases(sql: str, ctx: str) -> str:
    """
    Perbaiki nama kolom berdasar alias_map dan fallback classicmodels,
    TAPI jangan menyentuh teks dalam kutip ('...', "...", `...`)
    supaya alias/label buatan LLM tetap utuh.
    """
    alias_map_per_table = _extract_internal_hint(ctx)

    # global map (lower → canonical)
    global_map: Dict[str, Optional[str]] = {}
    for _t, amap in (alias_map_per_table or {}).items():
        for al, canon in (amap or {}).items():
            al = str(al).lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] != canon:
                global_map[al] = None

    # fallback untuk classicmodels
    classic_fallback = {
        "products": {
            "name": "productName",
            "productname": "productName",
            "listprice": "MSRP",
        },
        "orderdetails": {
            "listprice": "priceEach",
            "price": "priceEach",
        },
        "customers": {
            "name": "customerName",
            "customername": "customerName",
            "id": "customerNumber",
            "customerid": "customerNumber",
            "customer_id": "customerNumber",
            "id_customer": "customerNumber",
            "kodecustomer": "customerNumber",
            "kode_pelanggan": "customerNumber",
        },
    }
    for _t, amap in classic_fallback.items():
        for al, canon in amap.items():
            al = al.lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] is None:
                pass

    keywords = {
        "select","from","where","group","by","order","limit","join","left","right","inner",
        "on","and","or","not","as","distinct","count","sum","avg","min","max","having",
        "asc","desc","like","in","between","is","null","case","when","then","else","end"
    }
    token_pat = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\b")

    def replace_in_code_segment(seg: str) -> str:
        def smart_repl(m):
            w = m.group(0)
            lw = w.lower()
            if lw in keywords:
                return w
            canon = global_map.get(lw)
            if not canon:
                return w
            if not _ALNUM.match(canon):
                return w
            return canon
        return token_pat.sub(smart_repl, seg)

    parts = re.split(r"('(?:''|[^'])*'|\"(?:\"\"|[^\"])*\"|`[^`]*`)", sql)
    out_parts = []
    for idx, part in enumerate(parts):
        if idx % 2 == 1:
            out_parts.append(part)  # quoted → jangan disentuh
        else:
            out_parts.append(replace_in_code_segment(part))  # code → boleh direpair
    return "".join(out_parts)

def _looks_like_mask_example(s: str) -> bool:
    """
    Heuristik ringan untuk mendeteksi contoh pola yang lebih cocok REGEXP:
    - Mengandung kombinasi huruf + digit + pemisah ( -, /, _ )
      contoh: 'AAA-9999', 'AB-123', '2024-01-01'
    - Atau mengandung karakter placeholder umum: 9, #, X, N, A
    """
    if not s:
        return False

    # kalau terlalu pendek, biasanya bukan mask serius
    if len(s.strip()) < 4:
        return False

    # placeholder-style
    if any(ch in s for ch in ["9", "#", "X", "N", "A"]):
        return True

    # pola huruf2 + pemisah + digit2
    if re.search(r"[A-Za-z]{2,}[-_/][0-9]{2,}", s):
        return True

    # pola digit2 + pemisah + digit2
    if re.search(r"[0-9]{2,}[-_/][0-9]{2,}", s):
        return True

    return False


def _question_requires_regex(question: str) -> bool:
    q = (question or "").lower()

    # 0) eksplisit sebut regex
    if any(k in q for k in [" regexp ", "regex", "regular expression"]):
        return True

    def _mentions_code_like() -> bool:
        return any(k in q for k in ["code", "kode", "id", "identifier", "nomor", "no"])

    def _mentions_char_type() -> bool:
        return any(
            k in q
            for k in [
                "digit", "digits", "angka",
                "letter", "letters", "huruf",
                "alphanumeric", "alfanumerik",
            ]
        )

    # 1) exactly N digits/characters
    if re.search(r"\bexactly\s+\d+\s+(digit|digits|character|characters)\b", q):
        return True
    if re.search(r"\btepat\s+\d+\s+(digit|karakter|angka)\b", q):
        return True

    # 2) N-digit / N-letter
    if re.search(r"\b\d+\s*[- ]?(digit|digits|karakter|characters|letter|letters|huruf)\b", q):
        return True

    # 3) starts with ... followed by ... digits/letters
    if (
        "starts with" in q
        and "followed by" in q
        and _mentions_char_type()
    ):
        return True
    if (
        "diawali dengan" in q
        and "diikuti" in q
        and _mentions_char_type()
    ):
        return True

    # 4) only digits / only letters
    if any(
        k in q
        for k in [
            "digits only",
            "only digits",
            "numbers only",
            "only numbers",
            "hanya angka",
            "hanya berisi angka",
        ]
    ):
        return True

    if any(
        k in q
        for k in [
            "letters only",
            "only letters",
            "alphabetic only",
            "only alphabetic",
            "hanya huruf",
            "hanya berisi huruf",
        ]
    ):
        return True

    # 5) length = N + konteks code-like
    if (
        _mentions_code_like()
        and (
            ("length" in q and re.search(r"\blength\s*(=|==|is|adalah|:)\s*\d+\b", q))
            or re.search(r"\bpanjang(nya)?\s*(=|adalah)?\s*\d+\b", q)
            or re.search(r"\b(jumlah|count)\s+karakter\s*\d+\b", q)
        )
    ):
        return True

    # 6) format/pattern/mask + contoh yang kelihatan mask
    if any(k in q for k in ["format", "pattern", "pola", "mask"]):
        if _mentions_char_type() or _mentions_code_like():
            quoted = re.findall(r"'([^']*)'|\"([^\"]*)\"", question or "")
            examples = [a or b for (a, b) in quoted if (a or b)]
            if any(_looks_like_mask_example(s) for s in examples):
                return True

    # 7) "kode/code berbentuk/berformat ..."
    if _mentions_code_like() and any(k in q for k in ["berbentuk", "berformat", "like ", "seperti "]):
        quoted = re.findall(r"'([^']*)'|\"([^\"]*)\"", question or "")
        examples = [a or b for (a, b) in quoted if (a or b)]
        if _mentions_char_type() and examples:
            return True
        if any(_looks_like_mask_example(s) for s in examples):
            return True

    return False

# ---------- Prompt ----------
def _make_generation_prompt(ctx: str, question: str) -> str:
    return f"""
You are a senior SQL assistant for MySQL 8.0.

Return a runnable query using ONLY tables/columns listed in the ALLOWED IDENTIFIERS section of Context.
If a needed column name appears as an alias in Context, use its canonical column name.
If the query is about customers and the user does NOT explicitly ask for an id/number/code, prefer customerName over customerNumber AND also prefer customerName over contactFirstName/contactLastName when both are available in ALLOWED IDENTIFIERS.

IMPORTANT JOIN RULE:
- If all required columns live in a SINGLE table from ALLOWED IDENTIFIERS, DO NOT use any JOIN at all. Select only from that single table.
- Only introduce JOINs when you really need columns from more than one table.

PATTERN / REGEXP RULES (VERY IMPORTANT):
- When the user describes a string **pattern**, such as:
  - "starts with ..."
  - "ends with ..."
  - "contains ..."
  - "matches pattern ..."
  - "exactly N digits/characters"
  - "format ...", "mask ...", "only digits"/"hanya angka"
  you MUST use proper string pattern operators:
  - For structured or exact patterns (exact length, digit/letter structure, specific format),
    implement the core condition using REGEXP with anchors ^ and $.
  - You may use LIKE (or simple LEFT/RIGHT/SUBSTRING) only for naive wildcard cases
    where there is no requirement about exact length or digit/letter structure
    (e.g. "name starts with 'A'", "contains 'ABC'").
  - DO NOT try to emulate "exactly N digits/characters" using arithmetic tricks such as:
    expressions based on substrings compared with BETWEEN or numeric ranges.

########################
# STRICT OUTPUT FORMAT #
########################
{{ "sql": "<final MySQL query>", "notes": "<brief rationale; for EACH JOIN, name the exact selected/grouped/filtered column it contributes>" }}
# No extra keys. No Markdown. End SQL with a semicolon.
# If the final SQL has NO JOINs, set "notes" exactly to: "No joins used."

##################
# SQL CONSTRAINT #
##################
- MySQL 8.0; backticks only when needed; ANSI joins.

################
# CONTEXT
################
{ctx}

Question:
{question}
""".strip()

def _sanitize_notes(sql: str, notes: str) -> str:
    if not notes:
        return notes
    if not _has_join(sql or ""):
        return "No joins used."
    return notes.strip()

# ---------- extra helpers for generation robustness ----------
def _try_llm_json_three_times(llm: ChatOllama, base_prompt: str) -> Optional[Dict[str, Any]]:
    prompts = [
        base_prompt,
        base_prompt + "\n\nReturn ONLY the JSON object. Do not add explanation.",
        base_prompt + "\n\nReturn ONLY JSON with keys \"sql\" and \"notes\". No extra text. Example: {\"sql\":\"SELECT 1;\",\"notes\":\"No joins used.\"}",
    ]
    for p in prompts:
        resp = llm.invoke(p)
        content = getattr(resp, "content", None) or str(resp)
        try:
            obj = json.loads(content)
            if isinstance(obj, dict) and ("sql" in obj or "notes" in obj):
                return obj
        except Exception:
            m = re.search(r"\{.*\}", content, flags=re.S)
            if m:
                try:
                    obj = json.loads(m.group(0))
                    if isinstance(obj, dict) and ("sql" in obj or "notes" in obj):
                        return obj
                except Exception:
                    pass
    return None

def _build_llm(temp: float = 0.0) -> ChatOllama:
    """
    Helper standar untuk membuat LLM ChatOllama.
    Kalau nanti mau ganti model/param, cukup ubah di sini.
    """
    return ChatOllama(
        model=CHAT_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=temp,
    )


def _regenerate_with_extra_constraint(
    llm: ChatOllama,
    ctx: str,
    question: str,
    extra_msg: str,
) -> tuple[str, str]:
    """
    Helper untuk minta LLM regen SQL dengan constraint tambahan:
    - ctx akan digabung dengan extra_msg
    - return: (draft_sql_baru, notes_baru)
    """
    strict_prompt = _make_generation_prompt(ctx + "\n\n" + extra_msg, question)
    obj = _try_llm_json_three_times(llm, strict_prompt)
    if obj is None:
        return "", ""
    draft2 = (obj.get("sql") or "").strip()
    notes2 = (obj.get("notes") or "").strip()
    return draft2, notes2

# ---------- Multi-agent: Agent 1 (SQL Drafter) ----------
def draft_sql_agent(state: RAGState) -> RAGState:
    """
    Agent 1:
    - Mengambil question + context
    - Menentukan qtype & requires_regex (sekali saja, via helper)
    - Memanggil LLM sekali untuk membuat draft SQL + notes
    - Menyimpan hasil ke state['draft_sql'] dan state['notes']
    """
    llm = _build_llm()

    ctx = state.get("context", "")
    question = state.get("question", "")

    # DRY: pakai helper supaya turunan fitur selalu konsisten
    _ = _get_qtype(state)          # set state["qtype"] kalau belum ada
    _ = _get_requires_regex(state) # set state["requires_regex"] kalau belum ada

    base_prompt = _make_generation_prompt(ctx, question)

    draft_sql = ""
    notes = ""

    obj = _try_llm_json_three_times(llm, base_prompt)
    if obj is not None:
        draft_sql = (obj.get("sql") or "").strip()
        notes = (obj.get("notes") or "").strip()
    else:
        draft_sql = "SELECT 1;"
        notes = "Heuristic trivial fallback. No joins used."

    if draft_sql:
        _dbg_print_two("[Agent1 SQL] Draft (before post-processing):", draft_sql)

    # simpan hasil Agent 1 ke state, tapi belum dijadikan final answer.
    state["draft_sql"] = draft_sql
    state["notes"] = notes

    return state

# ---------- Multi-agent: Agent 2 (SQL Critic / Refiner) ----------
def refine_sql_agent(state: RAGState) -> RAGState:
    """
    Agent 2:
    - Ambil draft_sql + notes dari Agent 1
    - Jalankan heuristic pipeline (_normalize_sql_pipeline)
    - Kalau perlu, minta LLM regenerate dengan constraint tambahan:
        * single-table only (kalau ALLOWED cuma 1 tabel & qtype lookup)
        * REGEXP wajib (kalau requires_regex True)
        * whitelist identifer (ALLOWED IDENTIFIERS)
    - Simpan final_sql, notes yang sudah disanitasi, validation_errors, dan critic_feedback.
    """
    llm = _build_llm()

    ctx = state.get("context", "")
    question = state.get("question", "")
    draft_sql = (state.get("draft_sql") or "").strip()
    notes = state.get("notes") or ""
    qtype = _get_qtype(state)
    requires_regex = _get_requires_regex(state)

    corrections: list[str] = []  # NEW: catat apa yang diperbaiki agent 2

    if not draft_sql:
        # fallback kalau Agent 1 gagal total
        draft_sql = "SELECT 1;"
        corrections.append("Draft SQL missing, fallback to SELECT 1;")

    # 1) Normalisasi awal (postprocess + alias repair + postprocess)
    final_sql = _normalize_sql_pipeline(draft_sql, question, ctx)
    if final_sql:
        _dbg_print_two("[Agent2 SQL] After initial normalization:", final_sql)

    # 2) RULE: kalau ALLOWED cuma 1 tabel & qtype 'lookup' → paksa tanpa JOIN
    single_tbl = _get_single_allowed_table(ctx)
    if single_tbl and qtype == "lookup" and _has_join(final_sql or ""):
        extra = (
            f"The question is a simple lookup and ALLOWED IDENTIFIERS only lists a single table `{single_tbl}`.\n"
            "Rewrite the SQL so that it uses ONLY that single table and does NOT use any JOIN at all."
        )
        draft2, notes2 = _regenerate_with_extra_constraint(llm, ctx, question, extra)
        sql2 = _normalize_sql_pipeline(draft2 or "", question, ctx) if draft2 else ""
        if sql2 and not _has_join(sql2):
            corrections.append(
                f"Removed unnecessary JOINs because only single table `{single_tbl}` is allowed."
            )
            final_sql = sql2
            state["draft_sql"] = draft2 or draft_sql
            if notes2:
                notes = notes2

    # 3) RULE: enforce REGEXP untuk pertanyaan pattern-ketat
    if requires_regex and final_sql and "regexp" not in (final_sql or "").lower():
        extra = (
            "The question describes a strict string pattern (exact length and/or digit/character format). "
            "Rewrite the SQL so that this pattern is implemented using MySQL REGEXP with ^ and $ anchors. "
            "Do NOT use LIKE-only or numeric/substr tricks (e.g. RIGHT(...), CAST(...), BETWEEN ...) "
            "for the main pattern condition."
        )
        draft2, notes2 = _regenerate_with_extra_constraint(llm, ctx, question, extra)
        sql2 = _normalize_sql_pipeline(draft2 or "", question, ctx) if draft2 else ""
        if sql2 and "regexp" in sql2.lower():
            corrections.append("Enforced REGEXP usage for strict pattern requirement.")
            final_sql = sql2
            state["draft_sql"] = draft2 or draft_sql
            if notes2:
                notes = notes2

    # 4) RULE: whitelist check (ALLOWED IDENTIFIERS)
    violations = _validate_identifiers(final_sql or "", ctx)
    state["validation_errors"] = violations  # NEW: simpan list error pertama

    if violations:
        old_count = len(violations)

        extra = (
            "The previous SQL used identifiers not present in ALLOWED IDENTIFIERS:\n- "
            + "\n- ".join(violations)
            + "\nRewrite the SQL using ONLY identifiers listed in ALLOWED IDENTIFIERS. "
              "If a synonym appears, choose the canonical column."
        )
        draft2, notes2 = _regenerate_with_extra_constraint(llm, ctx, question, extra)
        sql2 = _normalize_sql_pipeline(draft2 or "", question, ctx) if draft2 else ""

        new_count = len(_validate_identifiers(sql2 or "", ctx))

        # HANYA terima kalau pelanggaran berkurang atau jadi 0
        if sql2 and (new_count == 0 or new_count < old_count):
            corrections.append(
                f"Reduced whitelist violations from {old_count} to {new_count}."
            )
            final_sql = sql2
            state["draft_sql"] = draft2 or draft_sql
            if notes2:
                notes = notes2
            # update violations ke hasil terakhir
            state["validation_errors"] = _validate_identifiers(final_sql or "", ctx)


    # 5) sanitasi notes berdasarkan ada/tidaknya JOIN
    notes = _sanitize_notes(final_sql or "", notes or "")

    # 6) simpan final result ke state
    if final_sql:
        answer_out = final_sql
    else:
        answer_out = draft_sql or "SELECT 1;"

    state["final_sql"] = final_sql or answer_out
    state["answer"] = answer_out
    state["notes"] = notes

    # NEW: simpan ringkasan pekerjaan Agent 2 (untuk defense / logging)
    if corrections:
        state["critic_feedback"] = " | ".join(corrections)
    else:
        state["critic_feedback"] = "No critical corrections from Agent 2."

    _dbg_print_two("[Agent2 SQL] Final SQL after all refinements:", state["final_sql"])

    return state

# =========================
# === APP GRAPH ===========
# =========================
def build_app():
    g = StateGraph(RAGState)

    # Existing nodes
    g.add_node("ingest", ingest_node)
    g.add_node("retrieve", retrieve_node)

    # NEW — Pisahkan multi-agent jadi node terpisah
    g.add_node("draft_sql", draft_sql_agent)     # NEW: Agent 1 as separate node
    g.add_node("refine_sql", refine_sql_agent)   # NEW: Agent 2 as separate node

    g.set_entry_point("ingest")

    # Existing flow
    g.add_edge("ingest", "retrieve")

    # NEW — Agent pipeline
    g.add_edge("retrieve", "draft_sql")   # NEW: draft SQL setelah retrieve
    g.add_edge("draft_sql", "refine_sql") # NEW: refine setelah draft
    g.add_edge("refine_sql", END)         # NEW: end setelah refinement

    return g.compile()

# =========================
# === MCP UTIL ============
# =========================
async def _mcp_call(client: Client, tool: str, payload: dict | None = None, timeout: int = MCP_TIMEOUT):
    coro = client.call_tool(tool, payload or {})
    return await asyncio.wait_for(coro, timeout=timeout)

def _extract_payload(result) -> Any:
    if hasattr(result, 'data') and result.data is not None:
        return result.data
    if hasattr(result, "structuredContent") and result.structuredContent is not None:
        return result.structuredContent
    if hasattr(result, "content") and result.content:
        return result.content[0].text if isinstance(result.content, list) else str(result.content)
    return result

def _ensure_json(d: Any) -> Optional[dict]:
    if isinstance(d, dict):
        return d
    if isinstance(d, str):
        try:
            return json.loads(d)
        except Exception:
            return None
    return None

def display_table_from_json(data, max_cell_len: int = 80):
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))

    def _shorten(val, max_len: int) -> str:
        # hanya potong string; biarkan tipe lain apa adanya
        if not isinstance(val, str):
            return val
        s = val.replace("\n", " ").strip()
        if len(s) <= max_len:
            return s
        return s[: max_len - 3] + "..."

    # apply truncate ke semua cell
    truncated_rows = [
        [_shorten(cell, max_cell_len) for cell in row]
        for row in preview_rows
    ]

    if truncated_rows and preview_cols:
        print("\nQuery Result (preview):")
        print(
            tabulate(
                truncated_rows,
                headers=preview_cols,
                tablefmt="github",
                showindex=range(1, len(truncated_rows) + 1),
            )
        )
        print(
            f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
            f"(full result: {total_rows} rows × {total_cols} columns)"
        )
    else:
        print("Query executed but returned no rows.")

# =========================
# === REPL RUNNER =========
# =========================
async def repl(paths: List[str]):
    app = build_app()  # NEW: gunakan LangGraph compiled app untuk orkestrasi penuh

    transport = StreamableHttpTransport(MCP_BASE)
    async with Client(transport=transport) as mcp:
        first_q = input("Ask me anything about your data: ").strip()
        if not first_q:
            print("No question entered. Exiting.")
            return

        t0_total = _now()  # start total timing untuk first run

        init: RAGState = {
            "jsonl_paths": paths,
            "question": first_q,
            "added": 0,
            "updated": 0,
            "skipped": 0,
            "context": "",
            "answer": "",
            "draft_sql": "",
            "final_sql": "",

            # NEW: default nilai untuk multi-agent
            "qtype": "",
 
            "notes": "",
            "validation_errors": [],
            "critic_feedback": "",
        }

        # NEW: jalankan seluruh pipeline via LangGraph:
        # ingest -> retrieve -> draft_sql -> refine_sql
        state = app.invoke(init)

        # NEW: gunakan hasil dari node 'ingest' di dalam graph sebagai ingestion report
        col = build_vector_store()
        total = col.count()
        line = _print_banner("Collection Ingestion Report")
        print(f"Added (first run): {state.get('added', 0)}")
        print(f"Updated (upserted): {state.get('updated', 0)}")
        print(f"Skipped (unchanged): {state.get('skipped', 0)}")
        print(f"Total docs now in collection: {total}")
        print(f"Chroma path: {Path(CHROMA_DIR).resolve()} | Collection: {COLLECTION_NAME}")
        print(f"{line}\n")

        async def exec_via_mcp(sql_block: str, final_sql: Optional[str] = None):
            # sql_block sekarang biasanya sudah hanya berisi SQL,
            # tapi tetap split untuk backward compatibility
            sql_only = sql_block.split("\n\n-- notes:")[0].strip()
            sql_clean = (final_sql or sql_only).rstrip(";").strip()

            # Log singkat & jelas: cuma query final yang dieksekusi
            print("\n[SQL EXEC] Final query sent to MCP:")
            print(sql_clean + ";")

            try:
                res = await _mcp_call(mcp, "query_db", {"sql": sql_clean})
                payload = _ensure_json(_extract_payload(res))
                if isinstance(payload, dict) and payload.get("success"):
                    display_table_from_json(payload)
                    return payload
                else:
                    err = payload.get("error") if isinstance(payload, dict) else str(payload)
                    print(f"query_db failed: {err}")
                    return None
            except asyncio.TimeoutError:
                print("MCP query_db timeout")
                return None
            except Exception as e:
                print(f"MCP query_db error: {e}")
                return None

        # eksekusi hasil pertanyaan pertama
        await exec_via_mcp(state.get("answer", ""), state.get("final_sql"))

        _log_total("End-to-end total time (first run)", t0_total)

        print("\nAsk me more questions! (type 'exit' to quit')")
        while True:
            q = input("\nQuestion> ").strip()
            if not q or q.lower() in {"exit", "quit", "q"}:
                print("Bye.")
                break

            t0_total = _now()

            # NEW: inisialisasi state baru untuk pertanyaan berikutnya
            state = {
                "jsonl_paths": paths,
                "question": q,
                "added": 0,
                "updated": 0,
                "skipped": 0,
                "context": "",
                "answer": "",
                "draft_sql": "",
                "final_sql": "",

                # NEW: reset field multi-agent untuk setiap Q baru
                "qtype": "",
                "notes": "",
                "validation_errors": [],
                "critic_feedback": "",
            }

            # NEW: jalankan ulang pipeline LangGraph
            # Catatan: ini juga akan menjalankan node 'ingest' lagi,
            # tetapi ingestion-mu sudah idempotent (pakai _canon_hash),
            # jadi baris yang sama akan di-skip sebagai "unchanged".
            state = app.invoke(state)

            await exec_via_mcp(state.get("answer", ""), state.get("final_sql"))

            _log_total("End-to-end total time (this question)", t0_total)

def main():
    paths = resolve_jsonl_paths(JSON_INPUT)
    if not paths:
        raise RuntimeError(
            "No JSONL inputs found.\n"
            "Set EXPORT_JSONL_OUTPUT to one of:\n"
            "  • a single file: ./exports/classicmodels.jsonl\n"
            "  • a directory:   ./datahub_metadata\n"
            "  • a glob:        ./datahub_metadata/*.jsonl\n"
        )

    try:
        asyncio.run(repl(paths))
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(repl(paths))
        else:
            raise

if __name__ == "__main__":
    main()

Ask me anything about your data:  Show the 10 products with the highest stock levels. Display the product code, name, and quantity in stock.



──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                    RETRIEVE DEBUG                                                                    
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
QUESTION      : Show the 10 products with the highest stock levels. Display the product code, name, and quantity in stock.
DENSE QUERY   : Show the 10 products with the highest stock levels. Display the product code, name, and quantity in stock. product productCode name
SPARSE QUERY  : 10 product highest stock level product code name quantity stock
TOKENS        : ['10', 'product', 'highest', 'stock', 'level', 'product', 'code', 'name', 'quantity', 'stock']
QTYPE         : lookup

SPARSE CANDIDATES (top 7 of 20):
       table          colum


Question>  List all orders placed in the year 2004. Show the order number and order date, ordered by order date.



──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                    RETRIEVE DEBUG                                                                    
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
QUESTION      : List all orders placed in the year 2004. Show the order number and order date, ordered by order date.
DENSE QUERY   : List all orders placed in the year 2004. Show the order number and order date, ordered by order date. order orderNumber
SPARSE QUERY  : order placed year 2004 order number order date ordered order date
TOKENS        : ['order', 'placed', 'year', '2004', 'order', 'number', 'order', 'date', 'ordered', 'order', 'date']
QTYPE         : lookup

SPARSE CANDIDATES (top 7 of 20):
       table          column         


Question>  Find all payments with an amount greater than 10,000. Show the customer number, check number, and amount, ordered by amount from highest to lowest.



──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                    RETRIEVE DEBUG                                                                    
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
QUESTION      : Find all payments with an amount greater than 10,000. Show the customer number, check number, and amount, ordered by amount from highest to lowest.
DENSE QUERY   : Find all payments with an amount greater than 10,000. Show the customer number, check number, and amount, ordered by amount from highest to lowest. customer customerName order orderNumber payments amount paymentDate
SPARSE QUERY  : payment greater 10000 customer number check number ordered highest lowest
TOKENS        : ['payment', 'greater', '10000', 'customer', 

... (7 outputs left out)


──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                    RETRIEVE DEBUG                                                                    
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
QUESTION      : Find employees whose phone extension starts with the letter "x" followed by exactly three digits. Show their employee numbers, first names, last names, and extensions.
DENSE QUERY   : Find employees whose phone extension starts with the letter "x" followed by exactly three digits. Show their employee numbers, first names, last names, and extensions. employee employeeNumber name
SPARSE QUERY  : employee phone extension start letter x followed exactly digit employee number name name extension
TOKENS        : ['employee', 'phon


Question>  Show orders that are either in status “Cancelled” or “On Hold”. Display the order number, order date, and status.



──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                    RETRIEVE DEBUG                                                                    
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
QUESTION      : Show orders that are either in status “Cancelled” or “On Hold”. Display the order number, order date, and status.
DENSE QUERY   : Show orders that are either in status “Cancelled” or “On Hold”. Display the order number, order date, and status. order orderNumber
SPARSE QUERY  : order statu cancelled hold order number order date statu
TOKENS        : ['order', 'statu', 'cancelled', 'hold', 'order', 'number', 'order', 'date', 'statu']
QTYPE         : lookup

SPARSE CANDIDATES (top 7 of 20):
       table          column         


Question>  Find products whose stock quantity is at least 500 and at most 1000 units. Show their product codes, names, and stock quantities.


## **CHECK COLLECTION**

In [7]:
from pprint import pprint
import chromadb
import numpy as np  # ok if installed; fallback guards below handle absence

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "classicmodels"

client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection(COLLECTION_NAME)

print("== BASIC INFO ==")
print("name :", col.name)
print("count:", col.count())

print("\n== FIRST 5 DOCS (with embeddings) ==")
res = col.get(limit=5, include=["metadatas", "documents", "embeddings"])

ids = res.get("ids", [])

embs = res.get("embeddings", None)
# normalize to a Python list of vectors or None
if embs is None:
    embs = []
else:
    # if it's a numpy array, convert to list; otherwise leave as-is
    try:
        # check without importing numpy explicitly
        import numpy as _np  # noqa
        if hasattr(embs, "tolist"):
            embs = embs.tolist()
    except Exception:
        # best-effort: if it still behaves like an array, wrap in list()
        try:
            embs = list(embs)
        except Exception:
            pass

n = min(len(ids), len(embs)) if embs else len(ids)

for i in range(n):
    id_ = ids[i]
    emb = embs[i] if i < len(embs) else None
    dim = None if emb is None else len(emb)
    print(f"{i+1:>2}. {id_} | emb_dim={dim}")

# count how many are None (only for the portion we fetched)
none_count = sum(1 for i in range(min(len(ids), len(embs))) if embs[i] is None) if embs else 0
print("Embeddings None (from first batch):", none_count)

# Optional: detail of the first doc
if ids:
    got_one = col.get(ids=[ids[0]], include=["metadatas", "documents", "embeddings"])
    print("\n== DETAIL DOC PERTAMA ==")
    pprint({k: got_one[k] for k in got_one if k in ("ids","metadatas","documents")})
    e0 = got_one.get("embeddings", [None])[0]
    if hasattr(e0, "tolist"):
        e0 = e0.tolist()
    print("emb_dim (doc[0]):", None if e0 is None else len(e0))

== BASIC INFO ==
name : classicmodels
count: 59

== FIRST 5 DOCS (with embeddings) ==
 1. urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)::customerNumber | emb_dim=1024
 2. urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)::customerName | emb_dim=1024
 3. urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)::contactLastName | emb_dim=1024
 4. urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)::contactFirstName | emb_dim=1024
 5. urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)::phone | emb_dim=1024
Embeddings None (from first batch): 0

== DETAIL DOC PERTAMA ==
{'documents': ['table: customers\n'
               'column: customerNumber\n'
               'description: Unique customer ID (primary key).\n'
               'aliases: customer id, customer number, id customer, kode '
               'customer, kode pelanggan\n'
               'roles: id'],
 'ids': ['urn:li:dataset:(urn:l

## **DELETE COLLECTION**

In [2]:
import chromadb

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "classicmodels"

client = chromadb.PersistentClient(path=CHROMA_DIR)

# cek dulu ada atau nggak
try:
    client.get_collection(COLLECTION_NAME)  # kalau gak ada bakal raise
    client.delete_collection(COLLECTION_NAME)
    print(f"[OK] collection '{COLLECTION_NAME}' deleted.")
except Exception as e:
    print(f"[INFO] collection '{COLLECTION_NAME}' not found or already deleted. ({e})")

[OK] collection 'classicmodels' deleted.
